In [6]:
# Cell 1 — Imports & Environment Check
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


# ── Standard library ──────────────────────────────────────────────
import os, json, re, ast, warnings, logging
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import defaultdict, Counter

# ── Scientific stack ──────────────────────────────────────────────
import numpy as np
import pandas as pd
from scipy.sparse import issparse

# ── sklearn ───────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.metrics import (f1_score, label_ranking_average_precision_score,
                             label_ranking_loss, coverage_error)
from sklearn.preprocessing import MultiLabelBinarizer

# ── skmultilearn (iterative split) ────────────────────────────────
try:
    from skmultilearn.model_selection import iterative_train_test_split
    HAS_SKMULTILEARN = True
    print("✓ skmultilearn available — will use iterative_train_test_split")
except ImportError:
    HAS_SKMULTILEARN = False
    print("⚠ skmultilearn not available — falling back to argmax stratification")

# ── PyTorch ───────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler          # legacy syntax (no device_type)

# ── HuggingFace ───────────────────────────────────────────────────
from transformers import (AutoTokenizer, AutoModel,
                          get_linear_schedule_with_warmup)

# ── Misc ──────────────────────────────────────────────────────────
import openpyxl                                           # xlsx reading
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

# ── Environment summary ───────────────────────────────────────────
print(f"\nPython  : {sys.version.split()[0]}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}  "
      f"({'  ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'})")
print(f"Transformers installed: ✓")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice  : {DEVICE}")

✓ skmultilearn available — will use iterative_train_test_split

Python  : 3.11.0
PyTorch : 2.11.0+cu126
CUDA    : True  (  NVIDIA GeForce RTX 3090)
Transformers installed: ✓

Device  : cuda


In [7]:
# Cell 2 — Seeds, Config & File Validation
import random
from types import SimpleNamespace

# ── Reproducibility ───────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# ── Central config ────────────────────────────────────────────────
CFG = SimpleNamespace(
    # --- Model ---
    encoder_name      = "ehsanaghaei/SecureBERT_Plus",
    max_len           = 512,
    freeze_layers     = 8,          # freeze first N encoder layers
    hidden_dim        = 512,
    dropout1          = 0.3,
    dropout2          = 0.2,

    # --- Loss (Asymmetric Loss) ---
    asl_gamma_neg     = 4,
    asl_gamma_pos     = 1,
    asl_clip          = 0.05,
    asl_clamp_max     = 10.0,

    # --- Class-weight clipping ---
    weight_min        = 0.5,
    weight_max        = 10.0,

    # --- Data ---
    min_support       = 10,         # drop techniques with <N training examples
    noisy_threshold   = 7,          # CVEs with >N techniques → noisy, weight 0.5
    noisy_weight      = 0.5,

    # --- Split ---
    train_frac        = 0.80,
    val_frac          = 0.10,
    test_frac         = 0.10,

    # --- Training ---
    batch_size        = 16,
    grad_accum        = 4,          # effective batch = 64
    learning_rate     = 1e-5,
    warmup_frac       = 0.10,
    max_epochs        = 15,
    patience          = 5,          # early-stop on val LRAP
    max_grad_norm     = 1.0,

    # --- DataLoader ---
    num_workers       = 0,
    pin_memory        = False,

    # --- Evaluation thresholds ---
    thresh_min        = 0.30,
    thresh_max        = 0.60,
    thresh_step       = 0.05,

    # --- Sentence-level max-pool inference ---
    sent_pool_enable  = True,       # toggle at eval time
    sent_min_chars    = 10,         # discard sentence chunks shorter than this
    sent_batch_size   = 64,         # how many sentences to encode per forward pass

    # --- Paths ---
    attack_json       = "OSRs/ATTACK/enterprise-attack-v16.1.json",
    cwe_xml           = "OSRs/MAPPINGS/cwec_latest.xml",
    capec_xml         = "OSRs/MAPPINGS/capec_latest.xml",
    kev_json          = "OSRs/kev-07.28.2025_attack-16.1-enterprise.json",
    smet_xlsx         = "CVE_annotated_dataset.xlsx",
    smet_id2mitre_url = "https://raw.githubusercontent.com/basel-a/SMET/main/id2mitre.json",
    nvd_pattern       = "OSRs/NVD/nvdcve-2.0-{year}.json",
    nvd_years         = list(range(2002, 2025)),
    checkpoint_path   = "best_model.pt",
)

# ── File validation ───────────────────────────────────────────────
required_files = {
    "ATT&CK STIX"  : CFG.attack_json,
    "CWE XML"      : CFG.cwe_xml,
    "CAPEC XML"    : CFG.capec_xml,
    "KEV gold"     : CFG.kev_json,
    "SMET xlsx"    : CFG.smet_xlsx,
}

print("── Required files ───────────────────────────────────────")
all_ok = True
for label, path in required_files.items():
    exists = Path(path).exists()
    size   = f"{Path(path).stat().st_size / 1e6:.1f} MB" if exists else "MISSING"
    status = "✓" if exists else "✗"
    print(f"  {status}  {label:<18} {path}  ({size})")
    if not exists:
        all_ok = False

print("\n── NVD JSON files ───────────────────────────────────────")
nvd_found, nvd_missing = [], []
for yr in CFG.nvd_years:
    p = Path(CFG.nvd_pattern.format(year=yr))
    (nvd_found if p.exists() else nvd_missing).append(yr)

print(f"  Found  : {len(nvd_found)} files  "
      f"({nvd_found[0]}–{nvd_found[-1]})" if nvd_found else "  Found  : 0 files")
if nvd_missing:
    print(f"  Missing: years {nvd_missing}")

CFG.nvd_years_available = nvd_found   # only iterate over years we actually have

print("\n── Config summary ───────────────────────────────────────")
print(f"  Encoder          : {CFG.encoder_name}")
print(f"  Freeze layers    : {CFG.freeze_layers}")
print(f"  Min support      : {CFG.min_support}")
print(f"  Batch / accum    : {CFG.batch_size} / {CFG.grad_accum}  → effective {CFG.batch_size*CFG.grad_accum}")
print(f"  ASL (γ-/γ+/clip) : {CFG.asl_gamma_neg}/{CFG.asl_gamma_pos}/{CFG.asl_clip}")
print(f"  Sent-pool infer  : {CFG.sent_pool_enable}  (min_chars={CFG.sent_min_chars}, batch={CFG.sent_batch_size})")

if not all_ok:
    raise FileNotFoundError("One or more required files are missing — fix paths before continuing.")
print("\n✓ All required files present. Ready for Cell 3.")

── Required files ───────────────────────────────────────
  ✓  ATT&CK STIX        OSRs/ATTACK/enterprise-attack-v16.1.json  (40.8 MB)
  ✓  CWE XML            OSRs/MAPPINGS/cwec_latest.xml  (16.1 MB)
  ✓  CAPEC XML          OSRs/MAPPINGS/capec_latest.xml  (3.8 MB)
  ✓  KEV gold           OSRs/kev-07.28.2025_attack-16.1-enterprise.json  (1.2 MB)
  ✓  SMET xlsx          CVE_annotated_dataset.xlsx  (0.1 MB)

── NVD JSON files ───────────────────────────────────────
  Found  : 23 files  (2002–2024)

── Config summary ───────────────────────────────────────
  Encoder          : ehsanaghaei/SecureBERT_Plus
  Freeze layers    : 8
  Min support      : 10
  Batch / accum    : 16 / 4  → effective 64
  ASL (γ-/γ+/clip) : 4/1/0.05
  Sent-pool infer  : True  (min_chars=10, batch=64)

✓ All required files present. Ready for Cell 3.


In [8]:
# Cell 2b — Path Fix: locate NVD 2.0 files & document schema differences
import glob

# ── Fix paths for files found in subdirectories ───────────────────
CFG.attack_json = "OSRs/ATTACK/enterprise-attack-v16.1.json"
CFG.cwe_xml     = "OSRs/MAPPINGS/cwec_latest.xml"
CFG.capec_xml   = "OSRs/MAPPINGS/capec_latest.xml"
CFG.kev_json    = "OSRs/kev-07.28.2025_attack-16.1-enterprise.json"

# ── NVD 2.0 schema config (replaces 1.1 field paths) ─────────────
# 2.0 structure per CVE entry (inside "vulnerabilities" array):
#   .cve.id                                          → CVE-ID
#   .cve.descriptions[{lang:"en"}].value             → description
#   .cve.weaknesses[*].description[*].value          → CWEs
#   .cve.metrics.cvssMetricV31[0].cvssData.*         → CVSS v3.1 (prefer)
#   .cve.metrics.cvssMetricV30[0].cvssData.*         → CVSS v3.0 (fallback)
#   .cve.metrics.cvssMetricV2[0].cvssData.*          → CVSS v2   (last resort)
CFG.nvd_version = "2.0"

# ── Locate NVD 2.0 JSON files ─────────────────────────────────────
search_globs = [
    "nvdcve-2.0-*.json",
    "*/nvdcve-2.0-*.json",
    "**/nvdcve-2.0-*.json",          # recursive
    "NVD/nvdcve-2.0-*.json",
    "OSRs/NVD/nvdcve-2.0-*.json",
    "nvd/nvdcve-2.0-*.json",
    "data/nvdcve-2.0-*.json",
]

found_nvd = {}
for pattern in search_globs:
    for fpath in sorted(glob.glob(pattern, recursive=True)):
        m = re.search(r'nvdcve-2\.0-(\d{4})\.json', fpath)
        if m:
            yr = int(m.group(1))
            if yr not in found_nvd:
                found_nvd[yr] = fpath

CFG.nvd_files_by_year   = found_nvd
CFG.nvd_years_available = sorted(found_nvd.keys())

print(f"NVD 2.0 files found: {len(found_nvd)}")
if found_nvd:
    years = CFG.nvd_years_available
    print(f"  Years : {years[0]}–{years[-1]}")
    dirs = sorted({str(Path(v).parent) for v in found_nvd.values()})
    for d in dirs:
        n = sum(1 for p in found_nvd.values() if str(Path(p).parent) == d)
        print(f"  Dir   : {d}  ({n} files)")
    # peek at one file to confirm schema
    sample_path = found_nvd[years[0]]
    with open(sample_path, encoding="utf-8") as f:
        sample = json.load(f)
    top_keys = list(sample.keys())
    n_vulns  = len(sample.get("vulnerabilities", []))
    print(f"\n  Schema check ({Path(sample_path).name}):")
    print(f"    Top-level keys : {top_keys}")
    print(f"    vulnerabilities: {n_vulns} entries")
    if n_vulns:
        first = sample["vulnerabilities"][0]["cve"]
        print(f"    First CVE id   : {first.get('id','?')}")
        print(f"    Metrics keys   : {list(first.get('metrics',{}).keys())}")
        print(f"    Weaknesses     : {first.get('weaknesses','none')[:1]}")
else:
    # Manual fallback
    print("  ✗ Not found automatically.")
    print("  Please populate CFG.nvd_files_by_year manually, e.g.:")
    print("    CFG.nvd_files_by_year = {2020: 'path/to/nvdcve-2.0-2020.json'}")

print("\nUpdated required paths:")
for label, path in [("ATT&CK STIX", CFG.attack_json),
                    ("CWE XML",     CFG.cwe_xml),
                    ("CAPEC XML",   CFG.capec_xml),
                    ("KEV gold",    CFG.kev_json),
                    ("SMET xlsx",   CFG.smet_xlsx)]:
    ok = Path(path).exists()
    print(f"  {'✓' if ok else '✗'}  {label:<14} {path}")

print("\n✓ Path fix done. Ready for Cell 3." if found_nvd else
      "\n⚠ Fix NVD paths manually then re-run before Cell 3.")

NVD 2.0 files found: 25
  Years : 2002–2026
  Dir   : OSRs\NVD  (25 files)

  Schema check (nvdcve-2.0-2002.json):
    Top-level keys : ['resultsPerPage', 'startIndex', 'totalResults', 'format', 'version', 'timestamp', 'vulnerabilities']
    vulnerabilities: 6770 entries
    First CVE id   : CVE-1999-0095
    Metrics keys   : ['cvssMetricV2']
    Weaknesses     : [{'source': 'nvd@nist.gov', 'type': 'Primary', 'description': [{'lang': 'en', 'value': 'NVD-CWE-Other'}]}]

Updated required paths:
  ✓  ATT&CK STIX    OSRs/ATTACK/enterprise-attack-v16.1.json
  ✓  CWE XML        OSRs/MAPPINGS/cwec_latest.xml
  ✓  CAPEC XML      OSRs/MAPPINGS/capec_latest.xml
  ✓  KEV gold       OSRs/kev-07.28.2025_attack-16.1-enterprise.json
  ✓  SMET xlsx      CVE_annotated_dataset.xlsx

✓ Path fix done. Ready for Cell 3.


In [9]:
# Cell 3 — ATT&CK Label Space (STIX parse)

with open(CFG.attack_json, encoding="utf-8") as f:
    stix_bundle = json.load(f)

objects = stix_bundle.get("objects", [])
print(f"Total STIX objects: {len(objects)}")

# ── Index all objects by id ───────────────────────────────────────
stix_by_id = {o["id"]: o for o in objects}

# ── 1. Collect ALL attack-pattern objects (techniques) ────────────
# External ref with source_name="mitre-attack" gives the T-code
def get_tcode(obj):
    for ref in obj.get("external_references", []):
        if ref.get("source_name") == "mitre-attack":
            return ref.get("external_id", "")
    return ""

all_techniques   = {}   # tcode → stix obj  (includes sub-techniques)
technique_names  = {}   # tcode → name

for o in objects:
    if o.get("type") != "attack-pattern":
        continue
    tc = get_tcode(o)
    if not tc.startswith("T"):
        continue
    all_techniques[tc]  = o
    technique_names[tc] = o.get("name", "")

print(f"Total attack-patterns (incl. subs): {len(all_techniques)}")

# ── 2. Build parent_map: sub-technique → parent ───────────────────
# Relationship type "subtechnique-of": source = sub, target = parent
parent_map = {}   # "T1059.001" → "T1059"

for o in objects:
    if o.get("type") != "relationship":
        continue
    if o.get("relationship_type") != "subtechnique-of":
        continue
    src_id  = o.get("source_ref", "")
    tgt_id  = o.get("target_ref", "")
    src_obj = stix_by_id.get(src_id)
    tgt_obj = stix_by_id.get(tgt_id)
    if not src_obj or not tgt_obj:
        continue
    sub_tc    = get_tcode(src_obj)
    parent_tc = get_tcode(tgt_obj)
    if sub_tc and parent_tc:
        parent_map[sub_tc] = parent_tc

print(f"Sub-technique → parent mappings: {len(parent_map)}")

# ── 3. Build revoked_map: revoked tcode → current tcode ──────────
# Relationship type "revoked-by": source = revoked, target = current
revoked_map = {}   # "Txxxx" → "Tyyyy"

for o in objects:
    if o.get("type") != "relationship":
        continue
    if o.get("relationship_type") != "revoked-by":
        continue
    src_obj = stix_by_id.get(o.get("source_ref", ""))
    tgt_obj = stix_by_id.get(o.get("target_ref", ""))
    if not src_obj or not tgt_obj:
        continue
    old_tc = get_tcode(src_obj)
    new_tc = get_tcode(tgt_obj)
    if old_tc and new_tc:
        revoked_map[old_tc] = new_tc

print(f"Revoked → current mappings       : {len(revoked_map)}")

# ── 4. Parent techniques only (no "." in ID, not revoked) ─────────
revoked_tcodes = set(revoked_map.keys())
# also mark objects with x_mitre_revoked=True
for tc, obj in all_techniques.items():
    if obj.get("x_mitre_revoked", False) or obj.get("revoked", False):
        revoked_tcodes.add(tc)

parent_techniques = {
    tc: obj for tc, obj in all_techniques.items()
    if "." not in tc and tc not in revoked_tcodes
}

print(f"Active parent techniques         : {len(parent_techniques)}")

# ── 5. Helper: resolve any tcode to its active parent ─────────────
def resolve_to_parent(tc):
    """sub→parent, revoked→current, then ensure no dot."""
    tc = revoked_map.get(tc, tc)       # un-revoke first
    tc = parent_map.get(tc, tc)        # sub → parent
    tc = revoked_map.get(tc, tc)       # parent itself might be revoked
    return tc

# ── 6. Build ordered label list (sorted for reproducibility) ──────
ALL_PARENT_TCODES = sorted(parent_techniques.keys())   # will be filtered later
print(f"\nFirst 10 parent T-codes: {ALL_PARENT_TCODES[:10]}")
print(f"Last  10 parent T-codes: {ALL_PARENT_TCODES[-10:]}")

# ── 7. CAPEC external refs on technique objects ───────────────────
# Some STIX technique objects carry capec refs directly
# capec_to_techniques: capec_id (str, no prefix) → set of parent tcodes
capec_to_techniques_stix = defaultdict(set)

for tc, obj in all_techniques.items():
    for ref in obj.get("external_references", []):
        src = ref.get("source_name", "").lower()
        if "capec" in src:
            raw = ref.get("external_id", "")
            cid = re.sub(r'[^\d]', '', raw)   # keep digits only → "112"
            if cid:
                parent_tc = resolve_to_parent(tc)
                if parent_tc in parent_techniques:
                    capec_to_techniques_stix[cid].add(parent_tc)

print(f"\nCAPEC→technique mappings (from STIX): {len(capec_to_techniques_stix)}")
print("Sample:", dict(list(capec_to_techniques_stix.items())[:3]))


Total STIX objects: 22905
Total attack-patterns (incl. subs): 799
Sub-technique → parent mappings: 456
Revoked → current mappings       : 139
Active parent techniques         : 214

First 10 parent T-codes: ['T1001', 'T1003', 'T1005', 'T1006', 'T1007', 'T1008', 'T1010', 'T1011', 'T1012', 'T1014']
Last  10 parent T-codes: ['T1650', 'T1651', 'T1652', 'T1653', 'T1654', 'T1656', 'T1657', 'T1659', 'T1665', 'T1666']

CAPEC→technique mappings (from STIX): 32
Sample: {'13': {'T1562'}, '17': {'T1574'}, '163': {'T1566'}}


In [11]:
# Cell 4a — CWE XML Diagnostics

cwe_tree = ET.parse(CFG.cwe_xml)
cwe_root = cwe_tree.getroot()

print(f"Root tag       : {cwe_root.tag}")
print(f"Root attribs   : {dict(list(cwe_root.attrib.items())[:5])}")

# Show first 2 levels of children
print("\nTop-level children:")
for i, child in enumerate(cwe_root):
    print(f"  [{i}] {child.tag}  attribs={dict(list(child.attrib.items())[:3])}")
    if i >= 5:
        print("  ...")
        break

# Dig into Weaknesses container
print("\nLooking for Weakness elements...")
all_tags = set()
for el in cwe_root.iter():
    all_tags.add(el.tag)

# print all unique tags (strip namespace for readability)
clean_tags = sorted({re.sub(r'\{[^}]+\}', '', t) for t in all_tags})
print(f"Unique tag names ({len(clean_tags)}): {clean_tags[:40]}")

# Find first Weakness element however it's tagged
weakness_el = None
for el in cwe_root.iter():
    if re.sub(r'\{[^}]+\}', '', el.tag) == "Weakness":
        weakness_el = el
        break

if weakness_el is not None:
    print(f"\nFirst Weakness tag  : {weakness_el.tag}")
    print(f"First Weakness attribs: {dict(list(weakness_el.attrib.items())[:6])}")
    # show its direct children tags
    child_tags = [re.sub(r'\{[^}]+\}','',c.tag) for c in weakness_el]
    print(f"Children tags       : {child_tags}")
    # look for any Related_Attack_Pattern-like child
    for c in weakness_el.iter():
        ctag = re.sub(r'\{[^}]+\}','',c.tag)
        if "attack" in ctag.lower() or "capec" in ctag.lower() or "related" in ctag.lower():
            print(f"\nFound relevant child: {c.tag}")
            print(f"  attribs: {dict(c.attrib)}")
            break
else:
    print("\n✗ No Weakness element found at all — check file integrity")
    # show raw first 500 chars
    with open(CFG.cwe_xml, encoding="utf-8") as f:
        print("\nRaw file head:\n", f.read(800))

Root tag       : {http://cwe.mitre.org/cwe-7}Weakness_Catalog
Root attribs   : {'Name': 'CWE', 'Version': '4.19.1', 'Date': '2026-01-21', '{http://www.w3.org/2001/XMLSchema-instance}schemaLocation': 'http://cwe.mitre.org/cwe-7 http://cwe.mitre.org/data/xsd/cwe_schema_v7.3.xsd'}

Top-level children:
  [0] {http://cwe.mitre.org/cwe-7}Weaknesses  attribs={}
  [1] {http://cwe.mitre.org/cwe-7}Categories  attribs={}
  [2] {http://cwe.mitre.org/cwe-7}Views  attribs={}
  [3] {http://cwe.mitre.org/cwe-7}External_References  attribs={}

Looking for Weakness elements...
Unique tag names (132): ['Affected_Resource', 'Affected_Resources', 'Alternate_Term', 'Alternate_Terms', 'Applicable_Platforms', 'Architecture', 'Audience', 'Author', 'Background_Detail', 'Background_Details', 'Body_Text', 'Categories', 'Category', 'Comments', 'Common_Consequences', 'Consequence', 'Content_History', 'Contribution', 'Contribution_Comment', 'Contribution_Date', 'Contribution_Name', 'Contribution_Organization', 'Cont

In [12]:
# Cell 4b — CWE XML Deep Diagnostics

NS7 = "http://cwe.mitre.org/cwe-7"

# ── 1. Full unique tag list ───────────────────────────────────────
all_tags_full = sorted({re.sub(r'\{[^}]+\}', '', el.tag) for el in cwe_root.iter()})
print(f"ALL unique tags ({len(all_tags_full)}):")
for i in range(0, len(all_tags_full), 5):
    print("  ", all_tags_full[i:i+5])

# ── 2. Hunt for any element whose tag OR attributes mention capec/attack ──
print("\nElements with 'capec' or 'attack' in tag or attribute keys/values:")
hits = 0
for el in cwe_root.iter():
    tag_clean = re.sub(r'\{[^}]+\}', '', el.tag).lower()
    attrib_str = str(el.attrib).lower()
    if "capec" in tag_clean or "capec" in attrib_str \
            or ("attack" in tag_clean and "pattern" in tag_clean):
        print(f"  tag={re.sub(r'{.*}','',el.tag)}  attribs={dict(el.attrib)}")
        if el.text and el.text.strip():
            print(f"    text={el.text.strip()[:80]}")
        hits += 1
        if hits >= 20:
            print("  ... (stopped at 20)")
            break

if hits == 0:
    print("  None found via tag/attrib scan.")

# ── 3. Check a CWE known to map to CAPEC (CWE-89 → CAPEC-66,7,110) ──
print("\nLooking for CWE-89 specifically:")
for el in cwe_root.iter(f"{{{NS7}}}Weakness"):
    if el.get("ID") == "89":
        print(f"  Found: {el.get('Name')}")
        for child in el:
            ctag = re.sub(r'\{[^}]+\}','', child.tag)
            print(f"    child: {ctag}  attribs={dict(list(child.attrib.items())[:4])}")
            # recurse one more level
            for grandchild in child:
                gctag = re.sub(r'\{[^}]+\}','', grandchild.tag)
                print(f"      grandchild: {gctag}  attribs={dict(list(grandchild.attrib.items())[:4])}")
                if grandchild.text and grandchild.text.strip():
                    print(f"        text: {grandchild.text.strip()[:60]}")
        break

ALL unique tags (132):
   ['Affected_Resource', 'Affected_Resources', 'Alternate_Term', 'Alternate_Terms', 'Applicable_Platforms']
   ['Architecture', 'Audience', 'Author', 'Background_Detail', 'Background_Details']
   ['Body_Text', 'Categories', 'Category', 'Comments', 'Common_Consequences']
   ['Consequence', 'Content_History', 'Contribution', 'Contribution_Comment', 'Contribution_Date']
   ['Contribution_Name', 'Contribution_Organization', 'Contribution_ReleaseDate', 'Contribution_Version', 'Demonstrative_Example']
   ['Demonstrative_Examples', 'Description', 'Detection_Method', 'Detection_Methods', 'Edition']
   ['Effectiveness', 'Effectiveness_Notes', 'Entry_ID', 'Entry_Name', 'Example_Code']
   ['Extended_Description', 'External_Reference', 'External_References', 'Filter', 'Functional_Area']
   ['Functional_Areas', 'Has_Member', 'Impact', 'Intro_Text', 'Introduction']
   ['Language', 'Likelihood', 'Likelihood_Of_Exploit', 'Link', 'Mapping_Fit']
   ['Mapping_Notes', 'Members', 'Me

In [13]:
# Cell 4 (fixed) — BRON Mapping: CWE→CAPEC→ATT&CK

NS7 = "http://cwe.mitre.org/cwe-7"

# ── 1. CWE XML: CWE → CAPEC (all Related_Attack_Pattern, no Nature filter) ──
cwe_to_capec = defaultdict(set)   # "CWE-79" → {"108","109",...}

for weakness in cwe_root.iter(f"{{{NS7}}}Weakness"):
    cwe_id = "CWE-" + weakness.get("ID", "")
    for rap in weakness.iter(f"{{{NS7}}}Related_Attack_Pattern"):
        capec_id = rap.get("CAPEC_ID", "").strip()
        if capec_id:
            cwe_to_capec[cwe_id].add(capec_id)

print(f"CWE→CAPEC mappings               : {len(cwe_to_capec)} CWEs")
total_pairs = sum(len(v) for v in cwe_to_capec.values())
print(f"Total (CWE, CAPEC) pairs         : {total_pairs}")
sample = list(cwe_to_capec.items())[:4]
print("Sample:", [(k, sorted(v)[:5]) for k, v in sample])

# ── 2. CAPEC XML: CAPEC → ATT&CK (Taxonomy_Name="ATTACK") ────────
capec_tree = ET.parse(CFG.capec_xml)
capec_root = capec_tree.getroot()

ns_c_match = re.match(r'\{.*\}', capec_root.tag)
NSC = ns_c_match.group(0) if ns_c_match else ""

capec_to_techniques_xml = defaultdict(set)   # "112" → {parent tcodes}

for ap in capec_root.iter(f"{NSC}Attack_Pattern"):
    capec_id = ap.get("ID", "").strip()
    for tm in ap.iter(f"{NSC}Taxonomy_Mapping"):
        if tm.get("Taxonomy_Name", "").upper() == "ATTACK":
            entry_el = tm.find(f"{NSC}Entry_ID")
            if entry_el is None or not entry_el.text:
                continue
            entry_id = entry_el.text.strip()
            if not entry_id.upper().startswith("T"):
                entry_id = "T" + entry_id
            entry_id = entry_id.upper()
            if re.match(r'T\d{4}', entry_id):
                parent_tc = resolve_to_parent(entry_id)
                if parent_tc in parent_techniques:
                    capec_to_techniques_xml[capec_id].add(parent_tc)

print(f"\nCAPEC→technique (XML)            : {len(capec_to_techniques_xml)} CAPECs")
print(f"Total (CAPEC, technique) pairs   : {sum(len(v) for v in capec_to_techniques_xml.values())}")

# ── 3. Merge CAPEC→technique (XML + STIX) ────────────────────────
capec_to_techniques = defaultdict(set)
for cid, techs in capec_to_techniques_xml.items():
    capec_to_techniques[cid].update(techs)
for cid, techs in capec_to_techniques_stix.items():
    capec_to_techniques[cid].update(techs)

print(f"\nMerged CAPEC→technique           : {len(capec_to_techniques)} CAPECs")
print(f"Total (CAPEC, technique) pairs   : {sum(len(v) for v in capec_to_techniques.values())}")

# ── 4. Build BRON mask: CWE → set of allowed parent techniques ───
cwe_to_techniques = defaultdict(set)

for cwe_id, capec_ids in cwe_to_capec.items():
    for cid in capec_ids:
        cwe_to_techniques[cwe_id].update(capec_to_techniques.get(cid, set()))

cwes_with_chain = sum(1 for v in cwe_to_techniques.values() if v)
cwes_no_chain   = len(cwe_to_capec) - cwes_with_chain
print(f"\nCWEs with ≥1 ATT&CK technique    : {cwes_with_chain}")
print(f"CWEs with no ATT&CK chain        : {cwes_no_chain}  (→ fallback: unconstrained)")
if cwes_with_chain:
    vals = [len(v) for v in cwe_to_techniques.values() if v]
    print(f"Avg techniques per CWE (chain>0) : {np.mean(vals):.1f}  (max={max(vals)})")

# ── 5. BRON mask lookup function ─────────────────────────────────
def get_bron_mask(cwe_list, label_index):
    """
    Returns bool array length len(label_index).
    All-True (unconstrained) if no CWE in the list has a chain.
    """
    allowed = set()
    for cwe in cwe_list:
        techs = cwe_to_techniques.get(cwe, set())
        if techs:
            allowed.update(techs)
    if not allowed:
        return np.ones(len(label_index), dtype=bool)
    mask = np.zeros(len(label_index), dtype=bool)
    for tc in allowed:
        if tc in label_index:
            mask[label_index[tc]] = True
    return mask

# ── 6. Spot checks ────────────────────────────────────────────────
print("\n── Spot checks ─────────────────────────────────────────")
for test_cwe in ["CWE-79", "CWE-89", "CWE-119", "CWE-20", "CWE-787"]:
    techs = sorted(cwe_to_techniques.get(test_cwe, set()))
    capecs = sorted(cwe_to_capec.get(test_cwe, set()))
    print(f"  {test_cwe:<12} {len(capecs):>3} CAPECs → {len(techs):>3} techniques  {techs[:6]}")

print("\n✓ BRON mapping complete. Ready for Cell 5.")

CWE→CAPEC mappings               : 336 CWEs
Total (CWE, CAPEC) pairs         : 1212
Sample: [('CWE-1007', ['632']), ('CWE-1021', ['103', '181', '222', '504', '506']), ('CWE-1037', ['663']), ('CWE-112', ['230', '231'])]

CAPEC→technique (XML)            : 177 CAPECs
Total (CAPEC, technique) pairs   : 238

Merged CAPEC→technique           : 179 CAPECs
Total (CAPEC, technique) pairs   : 242

CWEs with ≥1 ATT&CK technique    : 149
CWEs with no ATT&CK chain        : 187  (→ fallback: unconstrained)
Avg techniques per CWE (chain>0) : 3.0  (max=25)

── Spot checks ─────────────────────────────────────────
  CWE-79         6 CAPECs →   0 techniques  []
  CWE-89         6 CAPECs →   0 techniques  []
  CWE-119       12 CAPECs →   0 techniques  []
  CWE-20        51 CAPECs →   6 techniques  ['T1027', 'T1036', 'T1539', 'T1553', 'T1562', 'T1574']
  CWE-787        0 CAPECs →   0 techniques  []

✓ BRON mapping complete. Ready for Cell 5.


In [14]:
# Cell 5 — NVD 2.0 Parser: build raw training dataframe

SKIP_CWES = {"NVD-CWE-Other", "NVD-CWE-noinfo"}

def parse_cvss_v2(metric):
    """Extract vector string + score from cvssMetricV2 block."""
    cvss = metric.get("cvssData", {})
    vec  = cvss.get("vectorString", "")
    score = cvss.get("baseScore", None)
    # V2 vector: AV:N/AC:L/Au:N/C:P/I:P/A:P
    # Normalise to shorter tag form
    parts = {}
    for seg in vec.replace("(","").replace(")","").split("/"):
        if ":" in seg:
            k, v = seg.split(":", 1)
            parts[k] = v
    tag = (f"AV:{parts.get('AV','?')} AC:{parts.get('AC','?')} "
           f"Au:{parts.get('Au','?')} "
           f"C:{parts.get('C','?')} I:{parts.get('I','?')} A:{parts.get('A','?')} "
           f"SCORE:{score}")
    return tag.strip(), score

def parse_cvss_v3(metric):
    """Extract vector string + score from cvssMetricV3x block."""
    cvss  = metric.get("cvssData", {})
    vec   = cvss.get("vectorString", "")
    score = cvss.get("baseScore", None)
    # V3 vector: CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:H
    parts = {}
    for seg in vec.split("/"):
        if ":" in seg and not seg.startswith("CVSS"):
            k, v = seg.split(":", 1)
            parts[k] = v
    tag = (f"AV:{parts.get('AV','?')} AC:{parts.get('AC','?')} "
           f"PR:{parts.get('PR','?')} UI:{parts.get('UI','?')} "
           f"S:{parts.get('S','?')} C:{parts.get('C','?')} "
           f"I:{parts.get('I','?')} A:{parts.get('A','?')} "
           f"SCORE:{score}")
    return tag.strip(), score

def build_cvss_prefix(cve_metrics):
    """Return (prefix_string_or_None, score_or_None)."""
    if not cve_metrics:
        return None, None
    # prefer V3.1 > V3.0 > V2
    for key in ("cvssMetricV31", "cvssMetricV30"):
        lst = cve_metrics.get(key, [])
        if lst:
            return parse_cvss_v3(lst[0])
    lst = cve_metrics.get("cvssMetricV2", [])
    if lst:
        return parse_cvss_v2(lst[0])
    return None, None

def build_enriched_desc(raw_desc, cvss_prefix, cwes):
    parts = []
    if cvss_prefix:
        parts.append(f"[{cvss_prefix}]")
    if cwes:
        parts.append(f"[CWE: {', '.join(sorted(cwes))}]")
    parts.append(raw_desc)
    return " ".join(parts)

# ── Main parse loop ───────────────────────────────────────────────
records = []          # list of dicts
skipped_no_cwe   = 0
skipped_no_chain = 0
skipped_no_desc  = 0
total_seen       = 0

print("Parsing NVD 2.0 files...")
for yr in CFG.nvd_years_available:
    fpath = CFG.nvd_files_by_year[yr]
    with open(fpath, encoding="utf-8") as f:
        data = json.load(f)

    vulns = data.get("vulnerabilities", [])
    yr_kept = 0

    for entry in vulns:
        cve_obj = entry.get("cve", {})
        total_seen += 1

        # ── Description (English) ─────────────────────────────────
        desc = ""
        for d in cve_obj.get("descriptions", []):
            if d.get("lang", "") == "en":
                desc = d.get("value", "").strip()
                break
        if not desc or desc.startswith("** REJECT"):
            skipped_no_desc += 1
            continue

        cve_id = cve_obj.get("id", "")

        # ── CWEs ─────────────────────────────────────────────────
        cwes = set()
        for w in cve_obj.get("weaknesses", []):
            for wd in w.get("description", []):
                val = wd.get("value", "").strip()
                if val and val not in SKIP_CWES and val.startswith("CWE-"):
                    cwes.add(val)
        if not cwes:
            skipped_no_cwe += 1
            continue

        # ── BRON techniques ───────────────────────────────────────
        techniques = set()
        for cwe in cwes:
            techniques.update(cwe_to_techniques.get(cwe, set()))
        if not techniques:
            skipped_no_chain += 1
            continue

        # ── CVSS prefix ───────────────────────────────────────────
        cvss_prefix, cvss_score = build_cvss_prefix(cve_obj.get("metrics", {}))

        # ── Enriched description ──────────────────────────────────
        enriched = build_enriched_desc(desc, cvss_prefix, cwes)

        records.append({
            "cve_id"     : cve_id,
            "description": enriched,
            "raw_desc"   : desc,
            "cwes"       : list(cwes),
            "techniques" : list(techniques),
            "cvss_score" : cvss_score,
            "year"       : yr,
        })
        yr_kept += 1

    print(f"  {yr}: {len(vulns):>6} entries → {yr_kept:>5} kept")

print(f"\nTotal CVEs seen          : {total_seen:,}")
print(f"Skipped (no/reject desc) : {skipped_no_desc:,}")
print(f"Skipped (no valid CWE)   : {skipped_no_cwe:,}")
print(f"Skipped (no BRON chain)  : {skipped_no_chain:,}")
print(f"Kept                     : {len(records):,}")

df_raw = pd.DataFrame(records)
print(f"\ndf_raw shape: {df_raw.shape}")
print(df_raw[["cve_id","year","cvss_score","cwes","techniques"]].head(3).to_string())

Parsing NVD 2.0 files...
  2002:   6770 entries →   208 kept
  2003:   1555 entries →    96 kept
  2004:   2707 entries →    55 kept
  2005:   4769 entries →   141 kept
  2006:   7143 entries →   347 kept
  2007:   6580 entries →   766 kept
  2008:   7177 entries →  1509 kept
  2009:   5052 entries →   891 kept
  2010:   5244 entries →   875 kept
  2011:   4897 entries →  1083 kept
  2012:   5939 entries →  1019 kept
  2013:   6824 entries →  1389 kept
  2014:   9000 entries →  1574 kept
  2015:   8769 entries →  1875 kept
  2016:  10572 entries →  2858 kept
  2017:  17040 entries →  4428 kept
  2018:  17537 entries →  4420 kept
  2019:  17232 entries →  3877 kept
  2020:  21012 entries →  4754 kept
  2021:  23346 entries →  5463 kept
  2022:  27463 entries →  6251 kept
  2023:  31146 entries →  8025 kept
  2024:  38974 entries →  9990 kept
  2025:  42721 entries → 14178 kept
  2026:   3553 entries →  1242 kept

Total CVEs seen          : 333,022
Skipped (no/reject desc) : 0
Skipped (n

In [15]:
# Cell 5b — Rebuild enriched descriptions WITHOUT CWE tags
# Insert this cell, then rerun Cells 6 → 11 unchanged.

def build_enriched_desc_nocwe(raw_desc, cvss_prefix):
    """CVSS prefix only — no CWE tag to prevent leakage."""
    parts = []
    if cvss_prefix:
        parts.append(f"[{cvss_prefix}]")
    parts.append(raw_desc)
    return " ".join(parts)

# Rebuild df_raw descriptions in-place
# We need cvss_prefix per row — reparse from scratch using raw_desc + cvss_score
# Simpler: just strip the [CWE: ...] block from the existing enriched description
CWE_TAG_RE = re.compile(r'\[CWE:[^\]]+\]\s*')

before_sample = df_raw["description"].iloc[0]
df_raw["description"] = df_raw["description"].apply(
    lambda d: CWE_TAG_RE.sub("", d).strip())
after_sample = df_raw["description"].iloc[0]

print("CWE tag stripping:")
print(f"  Before: {before_sample[:120]}")
print(f"  After : {after_sample[:120]}")

# Verify no CWE tags remain
remaining = df_raw["description"].str.contains(r'\[CWE:', regex=True).sum()
print(f"\nRows still containing [CWE:] tag: {remaining}  "
      f"{'✓ clean' if remaining == 0 else '✗ check regex'}")

# Confirm CVSS prefix still present where expected
has_cvss = df_raw["description"].str.startswith("[AV:").sum()
no_prefix = (~df_raw["description"].str.startswith("[")).sum()
print(f"Rows with CVSS prefix            : {has_cvss:,}")
print(f"Rows with no prefix (no CVSS)    : {no_prefix:,}")

print(f"\nTotal rows                       : {len(df_raw):,}")
print("\n✓ Descriptions rebuilt. Now rerun Cells 6 → 11.")

CWE tag stripping:
  Before: [AV:L AC:L PR:N UI:N S:U C:H I:H A:H SCORE:8.4] [CWE: CWE-269] Certain NFS servers allow users to use mknod to gain priv
  After : [AV:L AC:L PR:N UI:N S:U C:H I:H A:H SCORE:8.4] Certain NFS servers allow users to use mknod to gain privileges by creat

Rows still containing [CWE:] tag: 0  ✓ clean
Rows with CVSS prefix            : 76,224
Rows with no prefix (no CVSS)    : 1,090

Total rows                       : 77,314

✓ Descriptions rebuilt. Now rerun Cells 6 → 11.


In [16]:
# Cell 6 — Data Cleaning, Label Filtering & Class Weights

# ── 1. Noisy flag: CVEs with >7 techniques ────────────────────────
df_raw["n_techniques"] = df_raw["techniques"].apply(len)
df_raw["noisy"]        = df_raw["n_techniques"] > CFG.noisy_threshold
df_raw["sample_weight"] = df_raw["noisy"].apply(
    lambda x: CFG.noisy_weight if x else 1.0)

print("── Noisy flag ───────────────────────────────────────────")
print(f"  Clean  (≤{CFG.noisy_threshold} techniques): "
      f"{(~df_raw['noisy']).sum():,}")
print(f"  Noisy  (>{CFG.noisy_threshold} techniques): "
      f"{df_raw['noisy'].sum():,}")

# ── 2. Deduplicate by raw description ────────────────────────────
# Keep row with most techniques per unique description
df_raw["desc_key"] = df_raw["raw_desc"].str.strip().str.lower()
before = len(df_raw)
df_raw = (df_raw
          .sort_values("n_techniques", ascending=False)
          .drop_duplicates(subset="desc_key", keep="first")
          .drop(columns=["desc_key"])
          .reset_index(drop=True))
print(f"\n── Deduplication ────────────────────────────────────────")
print(f"  Before: {before:,}  After: {len(df_raw):,}  "
      f"Dropped: {before-len(df_raw):,}")

# ── 3. Build initial label set (ALL parent techniques seen) ───────
all_seen_techniques = set()
for techs in df_raw["techniques"]:
    all_seen_techniques.update(techs)

print(f"\n── Initial label space ──────────────────────────────────")
print(f"  Unique techniques seen in data: {len(all_seen_techniques)}")

# ── 4. Min-support filtering ─────────────────────────────────────
# Count positive examples per technique
tech_counts = Counter()
for techs in df_raw["techniques"]:
    for t in techs:
        tech_counts[t] += 1

keep_techniques = {t for t, c in tech_counts.items()
                   if c >= CFG.min_support}
drop_techniques = all_seen_techniques - keep_techniques

print(f"\n── Min-support (≥{CFG.min_support}) ─────────────────────────────────")
print(f"  Techniques kept : {len(keep_techniques)}")
print(f"  Techniques dropped (rare): {len(drop_techniques)}")
if drop_techniques:
    rare_sample = sorted(drop_techniques)[:10]
    print(f"  Sample dropped  : {rare_sample}")

# ── 5. Filter technique lists & drop empty rows ───────────────────
df_raw["techniques"] = df_raw["techniques"].apply(
    lambda ts: [t for t in ts if t in keep_techniques])
df_raw["n_techniques"] = df_raw["techniques"].apply(len)

before = len(df_raw)
df_raw = df_raw[df_raw["n_techniques"] > 0].reset_index(drop=True)
print(f"\n  Rows dropped (all techniques removed): {before-len(df_raw):,}")
print(f"  Rows remaining                        : {len(df_raw):,}")

# ── 6. Final label index (sorted, reproducible) ───────────────────
LABEL_LIST  = sorted(keep_techniques)          # e.g. ['T1001','T1003',...]
LABEL_INDEX = {t: i for i, t in enumerate(LABEL_LIST)}
NUM_CLASSES = len(LABEL_LIST)
print(f"\n── Final label space ────────────────────────────────────")
print(f"  NUM_CLASSES : {NUM_CLASSES}")
print(f"  First 10    : {LABEL_LIST[:10]}")
print(f"  Last  10    : {LABEL_LIST[-10:]}")

# ── 7. MLB binarizer (fitted to LABEL_LIST) ───────────────────────
mlb = MultiLabelBinarizer(classes=LABEL_LIST)
mlb.fit([LABEL_LIST])    # single fit call to lock order
Y_full = mlb.transform(df_raw["techniques"].tolist())   # (N, NUM_CLASSES)
print(f"\n  Y_full shape : {Y_full.shape}  "
      f"density={Y_full.mean():.4f}")

# ── 8. Per-class weights: log(1 + N_neg/N_pos), clip [0.5, 10.0] ─
N = len(df_raw)
pos_counts  = Y_full.sum(axis=0).astype(float)          # (NUM_CLASSES,)
neg_counts  = N - pos_counts
# avoid div-by-zero for any class with 0 positives (shouldn't happen post filter)
pos_counts  = np.maximum(pos_counts, 1.0)
class_weights = np.log1p(neg_counts / pos_counts)
class_weights = np.clip(class_weights, CFG.weight_min, CFG.weight_max)

print(f"\n── Class weights ────────────────────────────────────────")
print(f"  Min   : {class_weights.min():.3f}")
print(f"  Max   : {class_weights.max():.3f}")
print(f"  Mean  : {class_weights.mean():.3f}")
print(f"  Median: {np.median(class_weights):.3f}")

# Store on CFG for use in loss
CFG.class_weights_np = class_weights

# ── 9. Quick label-distribution snapshot ─────────────────────────
print(f"\n── Top 10 most frequent techniques ──────────────────────")
top10 = sorted(zip(LABEL_LIST, pos_counts.astype(int)),
               key=lambda x: -x[1])[:10]
for tc, cnt in top10:
    name = technique_names.get(tc, "?")[:45]
    print(f"  {tc}  {cnt:>6}  {name}")

print(f"\n── Bottom 10 (near min-support) ─────────────────────────")
bot10 = sorted(zip(LABEL_LIST, pos_counts.astype(int)),
               key=lambda x: x[1])[:10]
for tc, cnt in bot10:
    name = technique_names.get(tc, "?")[:45]
    print(f"  {tc}  {cnt:>6}  {name}")

print("\n✓ Cleaning done. Ready for Cell 7.")

── Noisy flag ───────────────────────────────────────────
  Clean  (≤7 techniques): 55,330
  Noisy  (>7 techniques): 21,984

── Deduplication ────────────────────────────────────────
  Before: 77,314  After: 75,475  Dropped: 1,839

── Initial label space ──────────────────────────────────
  Unique techniques seen in data: 97

── Min-support (≥10) ─────────────────────────────────
  Techniques kept : 96
  Techniques dropped (rare): 1
  Sample dropped  : ['T1531']

  Rows dropped (all techniques removed): 4
  Rows remaining                        : 75,471

── Final label space ────────────────────────────────────
  NUM_CLASSES : 96
  First 10    : ['T1001', 'T1003', 'T1005', 'T1007', 'T1012', 'T1014', 'T1016', 'T1018', 'T1021', 'T1027']
  Last  10    : ['T1595', 'T1598', 'T1600', 'T1602', 'T1606', 'T1611', 'T1614', 'T1615', 'T1620', 'T1647']

  Y_full shape : (75471, 96)  density=0.0746

── Class weights ────────────────────────────────────────
  Min   : 0.621
  Max   : 8.929
  Mean  : 3

In [17]:
# Cell 7 — Train/Val/Test Split

# ── Inputs ────────────────────────────────────────────────────────
X_indices = np.arange(len(df_raw))          # just row indices
Y_sparse  = Y_full                           # (N, 96) dense numpy

# ── Strategy: iterative_train_test_split (skmultilearn) ──────────
# It expects scipy sparse matrices and 2-D X
from scipy.sparse import csr_matrix

if HAS_SKMULTILEARN:
    X_2d  = X_indices.reshape(-1, 1)
    Y_csr = csr_matrix(Y_sparse)

    # First split: train (80%) vs temp (20%)
    X_train_idx, Y_train, X_temp_idx, Y_temp = iterative_train_test_split(
        X_2d, Y_csr, test_size=CFG.val_frac + CFG.test_frac)

    # Second split: val (10%) vs test (10%)  → 50/50 of the temp 20%
    X_val_idx, Y_val, X_test_idx, Y_test = iterative_train_test_split(
        X_temp_idx, Y_temp, test_size=0.5)

    # Flatten back to 1-D index arrays
    train_idx = X_train_idx.flatten()
    val_idx   = X_val_idx.flatten()
    test_idx  = X_test_idx.flatten()

    # Convert sparse back to dense numpy
    Y_train = Y_train.toarray()
    Y_val   = Y_val.toarray()
    Y_test  = Y_test.toarray()

    split_method = "iterative_train_test_split"

else:
    # Fallback: stratify on argmax of label vector
    primary_label = Y_sparse.argmax(axis=1)

    train_idx, temp_idx = train_test_split(
        X_indices, test_size=CFG.val_frac + CFG.test_frac,
        stratify=primary_label, random_state=SEED)
    primary_temp = Y_sparse[temp_idx].argmax(axis=1)
    val_idx, test_idx = train_test_split(
        temp_idx, test_size=0.5,
        stratify=primary_temp, random_state=SEED)

    Y_train = Y_sparse[train_idx]
    Y_val   = Y_sparse[val_idx]
    Y_test  = Y_sparse[test_idx]

    split_method = "sklearn argmax stratify (fallback)"

# ── Subset dataframes ─────────────────────────────────────────────
df_train = df_raw.iloc[train_idx].reset_index(drop=True)
df_val   = df_raw.iloc[val_idx].reset_index(drop=True)
df_test  = df_raw.iloc[test_idx].reset_index(drop=True)

# ── Sanity checks ─────────────────────────────────────────────────
print(f"Split method : {split_method}")
print(f"\n  Train : {len(df_train):>6}  ({len(df_train)/len(df_raw)*100:.1f}%)")
print(f"  Val   : {len(df_val):>6}  ({len(df_val)/len(df_raw)*100:.1f}%)")
print(f"  Test  : {len(df_test):>6}  ({len(df_test)/len(df_raw)*100:.1f}%)")
print(f"  Total : {len(df_train)+len(df_val)+len(df_test):>6}")

# Label density per split
print(f"\n  Label density — train:{Y_train.mean():.4f}  "
      f"val:{Y_val.mean():.4f}  test:{Y_test.mean():.4f}")

# Per-class coverage: fraction of classes with ≥1 positive in each split
print(f"  Classes with ≥1 pos  — train:{(Y_train.sum(0)>0).sum()}  "
      f"val:{(Y_val.sum(0)>0).sum()}  test:{(Y_test.sum(0)>0).sum()}  "
      f"/ {NUM_CLASSES}")

# Noisy-row distribution
noisy_train = df_train["noisy"].mean()
noisy_val   = df_val["noisy"].mean()
noisy_test  = df_test["noisy"].mean()
print(f"  Noisy fraction       — train:{noisy_train:.3f}  "
      f"val:{noisy_val:.3f}  test:{noisy_test:.3f}")

# Check for leakage (no shared CVE IDs across splits)
ids_train = set(df_train["cve_id"])
ids_val   = set(df_val["cve_id"])
ids_test  = set(df_test["cve_id"])
leak_tv = ids_train & ids_val
leak_tt = ids_train & ids_test
leak_vt = ids_val   & ids_test
print(f"\n  Overlap train∩val  : {len(leak_tv)}")
print(f"  Overlap train∩test : {len(leak_tt)}")
print(f"  Overlap val∩test   : {len(leak_vt)}")

print("\n✓ Split done. Ready for Cell 8.")

Split method : iterative_train_test_split

  Train :  60484  (80.1%)
  Val   :   7489  (9.9%)
  Test  :   7498  (9.9%)
  Total :  75471

  Label density — train:0.0744  val:0.0753  test:0.0753
  Classes with ≥1 pos  — train:96  val:96  test:96  / 96
  Noisy fraction       — train:0.287  val:0.290  test:0.288

  Overlap train∩val  : 0
  Overlap train∩test : 0
  Overlap val∩test   : 0

✓ Split done. Ready for Cell 8.


In [18]:
# Cell 8 — Dataset, DataLoader & Sentence Splitter

# ── 1. Tokenizer ──────────────────────────────────────────────────
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(CFG.encoder_name)
print(f"✓ Tokenizer: {CFG.encoder_name}  vocab={tokenizer.vocab_size:,}")

# ── 2. Sentence splitter (for max-pool inference) ─────────────────
def split_sentences(text, min_chars=CFG.sent_min_chars):
    """
    Split text into sentences on '. ' or '; ', keep chunks ≥ min_chars.
    Returns list of strings (always includes the full text as last entry
    when called from inference — that's handled at call site).
    """
    # Split on '. ' or '; ' while preserving the delimiter at end of chunk
    raw = re.split(r'(?<=\.)\s+|(?<=;)\s+', text)
    chunks = [c.strip() for c in raw if len(c.strip()) >= min_chars]
    return chunks if chunks else [text]

# Quick test
_test = ("This vulnerability allows remote attackers to execute code. "
         "The issue exists in the parsing module; it fails to validate input. "
         "Affected versions include 1.0 through 3.2.")
print(f"\nSentence split test ({len(split_sentences(_test))} chunks):")
for s in split_sentences(_test):
    print(f"  [{len(s):3d}] {s}")

# ── 3. CVE Dataset ────────────────────────────────────────────────
class CVEDataset(Dataset):
    def __init__(self, df, labels, tokenizer, max_len):
        self.descriptions = df["description"].tolist()
        self.labels       = labels.astype(np.float32)
        self.weights      = df["sample_weight"].tolist()
        self.tokenizer    = tokenizer
        self.max_len      = max_len

    def __len__(self):
        return len(self.descriptions)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.descriptions[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids"     : enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels"        : torch.tensor(self.labels[idx], dtype=torch.float32),
            "sample_weight" : torch.tensor(self.weights[idx], dtype=torch.float32),
        }

# ── 4. Instantiate datasets ───────────────────────────────────────
ds_train = CVEDataset(df_train, Y_train, tokenizer, CFG.max_len)
ds_val   = CVEDataset(df_val,   Y_val,   tokenizer, CFG.max_len)
ds_test  = CVEDataset(df_test,  Y_test,  tokenizer, CFG.max_len)

print(f"\nDataset sizes — train:{len(ds_train)}  "
      f"val:{len(ds_val)}  test:{len(ds_test)}")

# ── 5. DataLoaders ────────────────────────────────────────────────
dl_train = DataLoader(
    ds_train,
    batch_size=CFG.batch_size,
    shuffle=True,
    num_workers=CFG.num_workers,
    pin_memory=CFG.pin_memory,
    drop_last=True,
)
dl_val = DataLoader(
    ds_val,
    batch_size=CFG.batch_size * 2,    # no grad → larger batch fine
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=CFG.pin_memory,
)
dl_test = DataLoader(
    ds_test,
    batch_size=CFG.batch_size * 2,
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=CFG.pin_memory,
)

print(f"Train batches/epoch : {len(dl_train)}")
print(f"Val   batches       : {len(dl_val)}")
print(f"Test  batches       : {len(dl_test)}")

# ── 6. Smoke-test one batch ───────────────────────────────────────
batch = next(iter(dl_train))
print(f"\nSmoke-test batch:")
print(f"  input_ids      : {batch['input_ids'].shape}")
print(f"  attention_mask : {batch['attention_mask'].shape}")
print(f"  labels         : {batch['labels'].shape}  "
      f"dtype={batch['labels'].dtype}")
print(f"  sample_weight  : {batch['sample_weight'].shape}  "
      f"values={batch['sample_weight'][:4].tolist()}")

print("\n✓ DataLoaders ready. Ready for Cell 9.")

Loading tokenizer...
✓ Tokenizer: ehsanaghaei/SecureBERT_Plus  vocab=50,265

Sentence split test (4 chunks):
  [ 59] This vulnerability allows remote attackers to execute code.
  [ 39] The issue exists in the parsing module;
  [ 27] it fails to validate input.
  [ 42] Affected versions include 1.0 through 3.2.

Dataset sizes — train:60484  val:7489  test:7498
Train batches/epoch : 3780
Val   batches       : 235
Test  batches       : 235

Smoke-test batch:
  input_ids      : torch.Size([16, 512])
  attention_mask : torch.Size([16, 512])
  labels         : torch.Size([16, 96])  dtype=torch.float32
  sample_weight  : torch.Size([16])  values=[1.0, 1.0, 1.0, 1.0]

✓ DataLoaders ready. Ready for Cell 9.


In [19]:
# Cell 9 — Model: SecureBERT_Plus + Classifier Head

# ── 1. Asymmetric Loss ────────────────────────────────────────────
class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05, clamp_max=10.0):
        super().__init__()
        self.gamma_neg  = gamma_neg
        self.gamma_pos  = gamma_pos
        self.clip       = clip
        self.clamp_max  = clamp_max

    def forward(self, logits, targets, class_weights=None, sample_weights=None):
        """
        logits        : (B, C)  raw logits
        targets       : (B, C)  float 0/1
        class_weights : (C,)    per-class pos_weight  [optional]
        sample_weights: (B,)    per-sample weight     [optional]
        """
        xs_pos = torch.sigmoid(logits)
        xs_neg = 1 - xs_pos

        # Asymmetric clip: shift negative probabilities down
        if self.clip > 0:
            xs_neg = (xs_neg + self.clip).clamp(max=1.0)

        lo_pos = torch.log(xs_pos.clamp(min=1e-8))
        lo_neg = torch.log(xs_neg.clamp(min=1e-8))

        loss = targets * lo_pos + (1 - targets) * lo_neg   # (B, C)

        # Asymmetric focusing
        if self.gamma_pos > 0 or self.gamma_neg > 0:
            pt_pos = xs_pos
            pt_neg = xs_neg
            gamma_t = (targets * self.gamma_pos
                       + (1 - targets) * self.gamma_neg)  # (B, C)
            pt      = targets * pt_pos + (1 - targets) * pt_neg
            loss    = loss * ((1 - pt) ** gamma_t)

        # Clamp to avoid runaway negatives
        loss = -loss.clamp(max=self.clamp_max)             # (B, C)

        # Per-class weighting
        if class_weights is not None:
            loss = loss * class_weights.unsqueeze(0)       # broadcast (1,C)

        loss = loss.sum(dim=1)                             # (B,)

        # Per-sample weighting
        if sample_weights is not None:
            loss = loss * sample_weights                   # (B,)

        return loss.mean()


# ── 2. Classifier model ───────────────────────────────────────────
class SecureBERTClassifier(nn.Module):
    def __init__(self, encoder_name, num_classes, hidden_dim=512,
                 dropout1=0.3, dropout2=0.2, freeze_layers=8):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(encoder_name)

        # Freeze embedding layer
        for param in self.encoder.embeddings.parameters():
            param.requires_grad = False

        # Freeze first `freeze_layers` transformer layers
        encoder_layers = self.encoder.encoder.layer
        for i, layer in enumerate(encoder_layers):
            if i < freeze_layers:
                for param in layer.parameters():
                    param.requires_grad = False

        d_model = self.encoder.config.hidden_size   # 768

        self.head = nn.Sequential(
            nn.Dropout(dropout1),
            nn.Linear(d_model, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout2),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids,
                           attention_mask=attention_mask)
        # [CLS] token representation
        cls = out.last_hidden_state[:, 0, :]    # (B, 768)
        return self.head(cls)                    # (B, num_classes)


# ── 3. Instantiate ────────────────────────────────────────────────
print("Loading encoder weights...")
model = SecureBERTClassifier(
    encoder_name  = CFG.encoder_name,
    num_classes   = NUM_CLASSES,
    hidden_dim    = CFG.hidden_dim,
    dropout1      = CFG.dropout1,
    dropout2      = CFG.dropout2,
    freeze_layers = CFG.freeze_layers,
).to(DEVICE)

# ── 4. Parameter audit ────────────────────────────────────────────
total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters()
                       if p.requires_grad)
frozen_params   = total_params - trainable_params

print(f"\nParameter audit:")
print(f"  Total      : {total_params:>12,}")
print(f"  Trainable  : {trainable_params:>12,}  "
      f"({trainable_params/total_params*100:.1f}%)")
print(f"  Frozen     : {frozen_params:>12,}  "
      f"({frozen_params/total_params*100:.1f}%)")

# Show which encoder layers are trainable
print(f"\nEncoder layer trainability (0–{len(model.encoder.encoder.layer)-1}):")
for i, layer in enumerate(model.encoder.encoder.layer):
    trainable = any(p.requires_grad for p in layer.parameters())
    status = "TRAIN" if trainable else "frozen"
    print(f"  Layer {i:>2}: {status}")

# ── 5. Loss & class-weight tensor ────────────────────────────────
criterion = AsymmetricLoss(
    gamma_neg = CFG.asl_gamma_neg,
    gamma_pos = CFG.asl_gamma_pos,
    clip      = CFG.asl_clip,
    clamp_max = CFG.asl_clamp_max,
)
class_weights_tensor = torch.tensor(
    CFG.class_weights_np, dtype=torch.float32).to(DEVICE)

# ── 6. Smoke-test forward pass ────────────────────────────────────
model.eval()
with torch.no_grad():
    _b = next(iter(dl_val))
    _logits = model(_b["input_ids"].to(DEVICE),
                    _b["attention_mask"].to(DEVICE))
    print(f"\nSmoke-test forward pass:")
    print(f"  Input  : {_b['input_ids'].shape}")
    print(f"  Logits : {_logits.shape}  "
          f"min={_logits.min():.3f}  max={_logits.max():.3f}")
    _loss = criterion(_logits, _b["labels"].to(DEVICE),
                      class_weights=class_weights_tensor)
    print(f"  ASL loss (random weights): {_loss.item():.4f}")

print("\n✓ Model ready. Ready for Cell 10.")

Loading encoder weights...


Loading weights: 100%|██████████| 197/197 [00:00<?, ?it/s]



Parameter audit:
  Total      :  125,089,632
  Trainable  :   29,386,080  (23.5%)
  Frozen     :   95,703,552  (76.5%)

Encoder layer trainability (0–11):
  Layer  0: frozen
  Layer  1: frozen
  Layer  2: frozen
  Layer  3: frozen
  Layer  4: frozen
  Layer  5: frozen
  Layer  6: frozen
  Layer  7: frozen
  Layer  8: TRAIN
  Layer  9: TRAIN
  Layer 10: TRAIN
  Layer 11: TRAIN

Smoke-test forward pass:
  Input  : torch.Size([32, 512])
  Logits : torch.Size([32, 96])  min=-0.940  max=0.826
  ASL loss (random weights): 38.8417

✓ Model ready. Ready for Cell 10.


In [20]:
# Cell 10 — Optimizer, Scheduler & Training Loop (with live output)
import sys
import time

# ── 1. Optimizer ──────────────────────────────────────────────────
optimizer = torch.optim.AdamW([
    {"params": [p for n, p in model.named_parameters()
                if p.requires_grad and "head" not in n],
     "lr": CFG.learning_rate},
    {"params": model.head.parameters(),
     "lr": CFG.learning_rate * 10},
], weight_decay=1e-2)

# ── 2. Scheduler ──────────────────────────────────────────────────
total_steps  = (len(dl_train) // CFG.grad_accum) * CFG.max_epochs
warmup_steps = int(total_steps * CFG.warmup_frac)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = warmup_steps,
    num_training_steps = total_steps,
)
print(f"Total optimizer steps : {total_steps:,}", flush=True)
print(f"Warmup steps          : {warmup_steps:,}", flush=True)

# ── 3. Scaler ─────────────────────────────────────────────────────
scaler = GradScaler()

# ── 4. Evaluation helper ──────────────────────────────────────────
@torch.no_grad()
def evaluate(model, dl, threshold=0.5):
    model.eval()
    all_logits, all_labels = [], []
    for batch in dl:
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        with autocast():
            logits = model(ids, mask)
        all_logits.append(logits.float().cpu())
        all_labels.append(batch["labels"].cpu())

    logits_np = torch.cat(all_logits).numpy()
    labels_np = torch.cat(all_labels).numpy()
    probs_np  = 1 / (1 + np.exp(-logits_np))
    preds     = (probs_np >= threshold).astype(int)

    lrap = label_ranking_average_precision_score(labels_np, probs_np)
    mif1 = f1_score(labels_np, preds, average="micro",    zero_division=0)
    maf1 = f1_score(labels_np, preds, average="macro",    zero_division=0)
    wf1  = f1_score(labels_np, preds, average="weighted", zero_division=0)

    return {"lrap": lrap, "micro_f1": mif1, "macro_f1": maf1,
            "weighted_f1": wf1, "logits": logits_np, "labels": labels_np}

# ── 5. Training loop ──────────────────────────────────────────────
best_val_lrap  = -1.0
patience_count = 0
history        = []
update_every   = 100   # print within-epoch progress every N steps

print(f"\nStarting training — {CFG.max_epochs} epochs, "
      f"patience={CFG.patience}", flush=True)

for epoch in range(1, CFG.max_epochs + 1):
    model.train()
    optimizer.zero_grad()

    running_loss = 0.0
    nan_skipped  = 0
    t0           = time.time()

    for step, batch in enumerate(dl_train):
        ids    = batch["input_ids"].to(DEVICE)
        mask   = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        sw     = batch["sample_weight"].to(DEVICE)

        with autocast():
            logits = model(ids, mask)
            loss   = criterion(logits, labels,
                               class_weights=class_weights_tensor,
                               sample_weights=sw)
            loss   = loss / CFG.grad_accum

        if not torch.isfinite(loss):
            nan_skipped += 1
            optimizer.zero_grad()
            continue

        scaler.scale(loss).backward()

        if (step + 1) % CFG.grad_accum == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), CFG.max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        running_loss += loss.item() * CFG.grad_accum

        # ── Live within-epoch progress ────────────────────────────
        if (step + 1) % update_every == 0 or (step + 1) == len(dl_train):
            avg_so_far = running_loss / (step + 1)
            pct        = (step + 1) / len(dl_train) * 100
            elapsed    = time.time() - t0
            eta        = elapsed / (step + 1) * (len(dl_train) - step - 1)
            print(f"\r  Ep {epoch:>2} [{step+1:>4}/{len(dl_train)}]"
                  f"  {pct:>5.1f}%"
                  f"  loss={avg_so_far:.4f}"
                  f"  nan_skip={nan_skipped}"
                  f"  ETA={eta:>5.0f}s",
                  end="", flush=True)

    avg_loss = running_loss / len(dl_train)
    epoch_time = time.time() - t0

    # ── Validation ────────────────────────────────────────────────
    print(f"\r  Ep {epoch:>2} — evaluating...           ",
          end="", flush=True)
    val_metrics = evaluate(model, dl_val, threshold=0.5)
    val_lrap    = val_metrics["lrap"]
    val_mif1    = val_metrics["micro_f1"]
    val_maf1    = val_metrics["macro_f1"]
    val_wf1     = val_metrics["weighted_f1"]

    is_best = val_lrap > best_val_lrap
    if is_best:
        best_val_lrap  = val_lrap
        patience_count = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "epoch"           : epoch,
            "best_lrap"       : best_val_lrap,
        }, CFG.checkpoint_path)
    else:
        patience_count += 1

    history.append({
        "epoch": epoch, "train_loss": avg_loss,
        "val_lrap": val_lrap, "val_micro_f1": val_mif1,
        "val_macro_f1": val_maf1, "nan_skipped": nan_skipped,
    })

    # ── Epoch summary line ────────────────────────────────────────
    best_marker = " ◀ BEST" if is_best else ""
    nan_warn    = f"  ⚠ nan_skip={nan_skipped}" if nan_skipped else ""
    print(f"\r┌─ Epoch {epoch:>2}/{CFG.max_epochs}"
          f"  time={epoch_time:.0f}s"
          f"  patience={patience_count}/{CFG.patience}{best_marker}",
          flush=True)
    print(f"│  train_loss={avg_loss:.4f}", flush=True)
    print(f"│  val  LRAP={val_lrap:.4f}"
          f"  micro-F1={val_mif1:.4f}"
          f"  macro-F1={val_maf1:.4f}"
          f"  weighted-F1={val_wf1:.4f}{nan_warn}", flush=True)
    print(f"└{'─'*55}", flush=True)

    if patience_count >= CFG.patience:
        print(f"\nEarly stop — no LRAP improvement for "
              f"{CFG.patience} epochs.", flush=True)
        break

print(f"\n✓ Training complete.", flush=True)
print(f"  Best val LRAP : {best_val_lrap:.4f}", flush=True)
print(f"  Checkpoint    : {CFG.checkpoint_path}", flush=True)

Total optimizer steps : 14,175
Warmup steps          : 1,417

Starting training — 15 epochs, patience=5
┌─ Epoch  1/15  time=294s  patience=0/5 ◀ BESTskip=0  ETA=    0ss
│  train_loss=6.1076
│  val  LRAP=0.7395  micro-F1=0.6225  macro-F1=0.3589  weighted-F1=0.6181
└───────────────────────────────────────────────────────
┌─ Epoch  2/15  time=296s  patience=0/5 ◀ BESTskip=0  ETA=    0s
│  train_loss=3.1411
│  val  LRAP=0.7862  micro-F1=0.6674  macro-F1=0.4317  weighted-F1=0.6661
└───────────────────────────────────────────────────────
┌─ Epoch  3/15  time=295s  patience=0/5 ◀ BESTskip=0  ETA=    0s
│  train_loss=2.7517
│  val  LRAP=0.8018  micro-F1=0.6824  macro-F1=0.4889  weighted-F1=0.6817
└───────────────────────────────────────────────────────
┌─ Epoch  4/15  time=296s  patience=0/5 ◀ BESTskip=0  ETA=    0s
│  train_loss=2.5781
│  val  LRAP=0.8081  micro-F1=0.6834  macro-F1=0.5028  weighted-F1=0.6828
└───────────────────────────────────────────────────────
┌─ Epoch  5/15  time=295s  

In [21]:
# Cell 11 — Load Best Checkpoint + Internal Test Evaluation

from sklearn.metrics import f1_score

# ── 1. Load best checkpoint ───────────────────────────────────────
ckpt = torch.load(CFG.checkpoint_path, map_location=DEVICE)
# handle both raw state_dict and wrapped dict
if "model_state_dict" in ckpt:
    model.load_state_dict(ckpt["model_state_dict"])
    print(f"✓ Loaded checkpoint from epoch {ckpt['epoch']}  "
          f"(val LRAP={ckpt['best_lrap']:.4f})")
else:
    model.load_state_dict(ckpt)
    print("✓ Loaded checkpoint (raw state_dict)")
model.eval()

# ── 2. Full inference on test set ────────────────────────────────
@torch.no_grad()
def get_probs(model, dl):
    all_logits, all_labels = [], []
    for batch in dl:
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        with autocast():
            logits = model(ids, mask)
        all_logits.append(logits.float().cpu())
        all_labels.append(batch["labels"].cpu())
    logits_np = torch.cat(all_logits).numpy()
    labels_np = torch.cat(all_labels).numpy()
    probs_np  = 1 / (1 + np.exp(-logits_np))
    return probs_np, labels_np

print("Running inference on test set...", flush=True)
test_probs, test_labels = get_probs(model, dl_test)
print(f"  probs shape : {test_probs.shape}")
print(f"  labels shape: {test_labels.shape}")

# ── 3. Recall@K helper ────────────────────────────────────────────
def recall_at_k(labels, probs, k):
    """Macro-averaged Recall@K."""
    top_k = np.argsort(-probs, axis=1)[:, :k]
    hits  = 0
    total = 0
    for i in range(len(labels)):
        pos = set(np.where(labels[i])[0])
        if not pos:
            continue
        hits  += len(pos & set(top_k[i]))
        total += len(pos)
    return hits / total if total > 0 else 0.0

# ── 4. Threshold sweep ────────────────────────────────────────────
print("\n── Threshold sweep (without BRON mask) ─────────────────")
thresholds = np.arange(CFG.thresh_min, CFG.thresh_max + 1e-9,
                       CFG.thresh_step)
sweep_results = []
print(f"  {'Thresh':>7}  {'MiF1':>7}  {'MaF1':>7}  {'WtF1':>7}  {'LRAP':>7}")
print(f"  {'─'*7}  {'─'*7}  {'─'*7}  {'─'*7}  {'─'*7}")
for t in thresholds:
    preds = (test_probs >= t).astype(int)
    mif1  = f1_score(test_labels, preds, average="micro",    zero_division=0)
    maf1  = f1_score(test_labels, preds, average="macro",    zero_division=0)
    wf1   = f1_score(test_labels, preds, average="weighted", zero_division=0)
    lrap  = label_ranking_average_precision_score(test_labels, test_probs)
    sweep_results.append((t, mif1, maf1, wf1, lrap))
    print(f"  {t:>7.2f}  {mif1:>7.4f}  {maf1:>7.4f}  {wf1:>7.4f}  {lrap:>7.4f}")

best_thresh = max(sweep_results, key=lambda x: x[1])[0]
print(f"\n  Best threshold (micro-F1): {best_thresh:.2f}")

# ── 5. BRON-masked inference ──────────────────────────────────────
def apply_bron_mask_batch(probs, df_subset, label_index):
    """Zero out logits for techniques outside BRON-allowed set."""
    masked = probs.copy()
    for i, (_, row) in enumerate(df_subset.iterrows()):
        mask = get_bron_mask(row["cwes"], label_index)
        masked[i][~mask] = 0.0
    return masked

print("\nApplying BRON mask to test set...", flush=True)
test_probs_masked = apply_bron_mask_batch(
    test_probs, df_test, LABEL_INDEX)
print("  Done.")

# ── 6. Full metrics: raw vs masked ───────────────────────────────
def full_metrics(probs, labels, threshold, tag=""):
    preds  = (probs >= threshold).astype(int)
    mif1   = f1_score(labels, preds, average="micro",    zero_division=0)
    maf1   = f1_score(labels, preds, average="macro",    zero_division=0)
    wf1    = f1_score(labels, preds, average="weighted", zero_division=0)
    lrap   = label_ranking_average_precision_score(labels, probs)
    rl     = label_ranking_loss(labels, probs)
    ce     = coverage_error(labels, probs)
    r1     = recall_at_k(labels, probs, 1)
    r3     = recall_at_k(labels, probs, 3)
    r5     = recall_at_k(labels, probs, 5)
    r10    = recall_at_k(labels, probs, 10)
    if tag:
        print(f"\n── {tag} (threshold={threshold:.2f}) ──────────────────")
    print(f"  micro-F1       : {mif1:.4f}")
    print(f"  macro-F1       : {maf1:.4f}")
    print(f"  weighted-F1    : {wf1:.4f}")
    print(f"  LRAP           : {lrap:.4f}")
    print(f"  Ranking Loss   : {rl:.4f}")
    print(f"  Coverage Error : {ce:.4f}")
    print(f"  R@1={r1:.4f}  R@3={r3:.4f}  R@5={r5:.4f}  R@10={r10:.4f}")
    return dict(micro_f1=mif1, macro_f1=maf1, weighted_f1=wf1,
                lrap=lrap, ranking_loss=rl, coverage_error=ce,
                r1=r1, r3=r3, r5=r5, r10=r10, threshold=threshold)

results_test_raw    = full_metrics(test_probs,        test_labels,
                                   best_thresh, "Internal test — NO mask")
results_test_masked = full_metrics(test_probs_masked, test_labels,
                                   best_thresh, "Internal test — BRON mask")

# ── 7. Training curve summary ─────────────────────────────────────
print("\n── Training history ─────────────────────────────────────")
print(f"  {'Ep':>3}  {'TrainLoss':>10}  {'ValLRAP':>9}  {'MiF1':>7}")
for h in history:
    print(f"  {h['epoch']:>3}  {h['train_loss']:>10.4f}  "
          f"{h['val_lrap']:>9.4f}  {h['val_micro_f1']:>7.4f}")

# Store for final summary cell
CFG._results = getattr(CFG, "_results", {})
CFG._results["internal_raw"]    = results_test_raw
CFG._results["internal_masked"] = results_test_masked
CFG._results["best_threshold"]  = best_thresh

print(f"\n⚠  Note: high internal metrics are expected — labels are derived")
print(f"   from the same BRON chain used to enrich descriptions.")
print(f"   True generalization will show on KEV & SMET evaluations.")
print("\n✓ Internal evaluation done. Ready for Cell 12.")


✓ Loaded checkpoint from epoch 15  (val LRAP=0.8214)
Running inference on test set...
  probs shape : (7498, 96)
  labels shape: (7498, 96)

── Threshold sweep (without BRON mask) ─────────────────
   Thresh     MiF1     MaF1     WtF1     LRAP
  ───────  ───────  ───────  ───────  ───────
     0.30   0.4372   0.3097   0.5093   0.8237
     0.35   0.5155   0.3726   0.5646   0.8237
     0.40   0.5901   0.4357   0.6173   0.8237
     0.45   0.6507   0.4821   0.6618   0.8237
     0.50   0.6956   0.5134   0.6975   0.8237
     0.55   0.7197   0.5284   0.7164   0.8237
     0.60   0.7163   0.5205   0.7101   0.8237

  Best threshold (micro-F1): 0.55

Applying BRON mask to test set...
  Done.

── Internal test — NO mask (threshold=0.55) ──────────────────
  micro-F1       : 0.7197
  macro-F1       : 0.5284
  weighted-F1    : 0.7164
  LRAP           : 0.8237
  Ranking Loss   : 0.0387
  Coverage Error : 12.2706
  R@1=0.1139  R@3=0.2808  R@5=0.3939  R@10=0.5780

── Internal test — BRON mask (threshol

In [24]:
# Cell 12 — KEV/CTID Gold-Set Evaluation

# ── 1. Load KEV file ──────────────────────────────────────────────
with open(CFG.kev_json, encoding="utf-8") as f:
    kev_data = json.load(f)

print(f"KEV file keys: {list(kev_data.keys())[:8]}")

# Discover structure — could be a list or a dict with a key
if isinstance(kev_data, list):
    kev_entries = kev_data
elif isinstance(kev_data, dict):
    # find the key that holds a list of CVE entries
    kev_entries = None
    for k, v in kev_data.items():
        if isinstance(v, list) and len(v) > 0:
            kev_entries = v
            print(f"  Using key '{k}'  ({len(v)} entries)")
            break
    if kev_entries is None:
        raise ValueError("Cannot find list of entries in KEV JSON")
else:
    raise ValueError(f"Unexpected KEV JSON type: {type(kev_data)}")

print(f"Total KEV entries: {len(kev_entries)}")
print(f"First entry keys : {list(kev_entries[0].keys())}")
print(f"First entry sample:\n  {kev_entries[0]}")

KEV file keys: ['metadata', 'mapping_objects']
  Using key 'mapping_objects'  (1183 entries)
Total KEV entries: 1183
First entry keys : ['capability_id', 'capability_description', 'mapping_type', 'attack_object_id', 'attack_object_name', 'capability_group', 'comments', 'references', 'status', 'mapping_framework_version']
First entry sample:
  {'capability_id': 'CVE-2024-34102', 'capability_description': 'Adobe Commerce and Magento Open Source Improper Restriction of XML External Entity Reference (XXE) Vulnerability', 'mapping_type': 'exploitation_technique', 'attack_object_id': 'T1190', 'attack_object_name': 'Exploit Public-Facing Application', 'capability_group': 'xxe', 'comments': 'This vulnerability is exploited by sending a crafted XML document that references external entities with the likely goal of accessing local data.', 'references': ['https://www.assetnote.io/resources/research/why-nested-deserialization-is-harmful-magento-xxe-cve-2024-34102', 'https://www.vicarius.io/vsociet

In [25]:
# Cell 12 (continued) — KEV/CTID Full Evaluation

# ── 2. Group by CVE → set of parent techniques ───────────────────
kev_by_cve = defaultdict(set)   # cve_id → set of parent tcodes
kev_desc   = {}                  # cve_id → description (for tokenization)

for entry in kev_entries:
    cve_id = entry.get("capability_id", "").strip()
    tc_raw = entry.get("attack_object_id", "").strip()
    desc   = entry.get("capability_description", "").strip()
    if not cve_id.startswith("CVE-") or not tc_raw:
        continue
    # resolve sub→parent, revoked→current
    parent_tc = resolve_to_parent(tc_raw)
    if parent_tc in LABEL_INDEX:
        kev_by_cve[cve_id].add(parent_tc)
    if desc and cve_id not in kev_desc:
        kev_desc[cve_id] = desc

print(f"Unique CVEs in KEV              : {len(kev_by_cve)}")
print(f"CVEs with ≥1 label in our space : "
      f"{sum(1 for v in kev_by_cve.values() if v)}")

# Techniques not in our label space
all_kev_raw = {e.get('attack_object_id','') for e in kev_entries}
outside = {t for t in all_kev_raw
           if resolve_to_parent(t) not in LABEL_INDEX and t}
print(f"Unique T-codes in KEV           : {len(all_kev_raw)}")
print(f"T-codes outside our label space : {sorted(outside)[:10]}")

# ── 3. Build KEV dataframe ────────────────────────────────────────
kev_rows = []
for cve_id, techs in kev_by_cve.items():
    if not techs:
        continue
    # get CWEs from training data if available (for BRON mask)
    train_row = df_raw[df_raw["cve_id"] == cve_id]
    if len(train_row) > 0:
        cwes     = train_row.iloc[0]["cwes"]
        desc_use = train_row.iloc[0]["description"]   # enriched
        in_train = True
    else:
        cwes     = []
        desc_use = kev_desc.get(cve_id, "")
        in_train = False
    kev_rows.append({
        "cve_id"    : cve_id,
        "description": desc_use,
        "cwes"      : cwes,
        "techniques": list(techs),
        "in_train"  : in_train,
    })

df_kev = pd.DataFrame(kev_rows)
print(f"\nKEV eval rows                   : {len(df_kev)}")
print(f"  Overlap with training set     : {df_kev['in_train'].sum()}")
print(f"  Exclusive (not in train)      : {(~df_kev['in_train']).sum()}")

# ── 4. Tokenize & run inference ───────────────────────────────────
class SimpleDataset(Dataset):
    def __init__(self, descriptions, tokenizer, max_len):
        self.descriptions = descriptions
        self.tokenizer    = tokenizer
        self.max_len      = max_len
    def __len__(self):
        return len(self.descriptions)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.descriptions[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {"input_ids"     : enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0)}

@torch.no_grad()
def infer_probs(model, descriptions, batch_size=32):
    ds  = SimpleDataset(descriptions, tokenizer, CFG.max_len)
    dl  = DataLoader(ds, batch_size=batch_size, shuffle=False,
                     num_workers=0, pin_memory=False)
    all_logits = []
    for batch in dl:
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        with autocast():
            logits = model(ids, mask)
        all_logits.append(logits.float().cpu())
    logits_np = torch.cat(all_logits).numpy()
    return 1 / (1 + np.exp(-logits_np))

print("\nRunning inference on KEV set...", flush=True)
model.eval()
kev_probs = infer_probs(model, df_kev["description"].tolist())
kev_labels = mlb.transform(df_kev["techniques"].tolist()).astype(np.float32)
print(f"  kev_probs shape  : {kev_probs.shape}")
print(f"  kev_labels shape : {kev_labels.shape}")
print(f"  kev_labels density: {kev_labels.mean():.4f}")

# ── 5. BRON-masked probs ──────────────────────────────────────────
kev_probs_masked = apply_bron_mask_batch(kev_probs, df_kev, LABEL_INDEX)

# ── 6. Metrics function (KEV subset) ─────────────────────────────
def kev_metrics(probs, labels, threshold, tag):
    preds = (probs >= threshold).astype(int)
    mif1  = f1_score(labels, preds, average="micro",    zero_division=0)
    maf1  = f1_score(labels, preds, average="macro",    zero_division=0)
    lrap  = label_ranking_average_precision_score(labels, probs)
    r5    = recall_at_k(labels, probs, 5)
    print(f"\n── KEV {tag} (threshold={threshold:.2f}) ───────────────")
    print(f"  micro-F1 : {mif1:.4f}   macro-F1 : {maf1:.4f}")
    print(f"  LRAP     : {lrap:.4f}   R@5       : {r5:.4f}")
    return dict(micro_f1=mif1, macro_f1=maf1, lrap=lrap, r5=r5)

t = CFG._results["best_threshold"]

# Full KEV
r_kev_raw    = kev_metrics(kev_probs,        kev_labels, t,
                            "full — NO mask")
r_kev_masked = kev_metrics(kev_probs_masked, kev_labels, t,
                            "full — BRON mask")

# Exclusive only (not in training set)
excl_mask = ~df_kev["in_train"].values
if excl_mask.sum() > 0:
    r_kev_excl_raw    = kev_metrics(kev_probs[excl_mask],
                                    kev_labels[excl_mask], t,
                                    "exclusive — NO mask")
    r_kev_excl_masked = kev_metrics(kev_probs_masked[excl_mask],
                                    kev_labels[excl_mask], t,
                                    "exclusive — BRON mask")
else:
    print("  (no exclusive CVEs — all KEV overlap with training)")
    r_kev_excl_raw = r_kev_excl_masked = None

# ── 7. Store results ──────────────────────────────────────────────
CFG._results["kev_raw"]        = r_kev_raw
CFG._results["kev_masked"]     = r_kev_masked
CFG._results["kev_excl_raw"]   = r_kev_excl_raw
CFG._results["kev_excl_masked"]= r_kev_excl_masked

print("\n✓ KEV evaluation done. Ready for Cell 13.")

Unique CVEs in KEV              : 248
CVEs with ≥1 label in our space : 248
Unique T-codes in KEV           : 155
T-codes outside our label space : ['T1011', 'T1041', 'T1047', 'T1048', 'T1048.003', 'T1053', 'T1053.005', 'T1059', 'T1059.001', 'T1059.003']

KEV eval rows                   : 248
  Overlap with training set     : 75
  Exclusive (not in train)      : 173

Running inference on KEV set...
  kev_probs shape  : (248, 96)
  kev_labels shape : (248, 96)
  kev_labels density: 0.0180

── KEV full — NO mask (threshold=0.55) ───────────────
  micro-F1 : 0.0475   macro-F1 : 0.0300
  LRAP     : 0.0797   R@5       : 0.0816

── KEV full — BRON mask (threshold=0.55) ───────────────
  micro-F1 : 0.0411   macro-F1 : 0.0220
  LRAP     : 0.0678   R@5       : 0.0816

── KEV exclusive — NO mask (threshold=0.55) ───────────────
  micro-F1 : 0.0392   macro-F1 : 0.0153
  LRAP     : 0.0592   R@5       : 0.0685

── KEV exclusive — BRON mask (threshold=0.55) ───────────────
  micro-F1 : 0.0392   macr

In [27]:
# Cell 13 — SMET 303 Benchmark Evaluation

# ── 1. Load SMET xlsx ─────────────────────────────────────────────
df_smet = pd.read_excel(CFG.smet_xlsx, engine="openpyxl")
print(f"SMET shape : {df_smet.shape}")
print(f"Columns    : {list(df_smet.columns)}")
print(f"\nSample row:")
print(df_smet.iloc[0].to_string())
print(f"\nATT&CK Techniques sample values:")
for v in df_smet.iloc[:3, df_smet.columns.get_loc(
        [c for c in df_smet.columns if 'echnique' in c or 'ttack' in c][0])]:
    print(f"  {repr(v)[:120]}")

SMET shape : (303, 4)
Columns    : ['ID', 'Description', 'ATT&CK Techniques', 'Manually extracted attack vectors']

Sample row:
ID                                                                      CVE-2021-29665
Description                          IBM Security Verify Access 20.07 is vulnerable...
ATT&CK Techniques                            ['Exploitation for Privilege Escalation']
Manually extracted attack vectors    ['execute arbitrary code on the system with el...

ATT&CK Techniques sample values:
  "['Exploitation for Privilege Escalation']"
  "['Exploitation for Privilege Escalation']"
  "['Endpoint Denial of Service']"


In [28]:
# Cell 13 — SMET 303 Benchmark Evaluation

# ── 1. Build name → T-code map from STIX ─────────────────────────
# Lowercase name → parent tcode (only techniques in our label space)
name_to_tcode = {}
for tc, obj in parent_techniques.items():
    if tc not in LABEL_INDEX:
        continue
    name = obj.get("name", "").strip().lower()
    if name:
        name_to_tcode[name] = tc

# Also add aliases from technique_names for any edge cases
for tc, name in technique_names.items():
    parent_tc = resolve_to_parent(tc)
    if parent_tc in LABEL_INDEX:
        name_to_tcode[name.strip().lower()] = parent_tc

print(f"Name→T-code entries : {len(name_to_tcode)}")
print("Sample:")
for k, v in list(name_to_tcode.items())[:5]:
    print(f"  '{k}' → {v}")

# ── 2. Try to fetch id2mitre.json as fallback ─────────────────────
id2mitre = {}
try:
    import urllib.request
    with urllib.request.urlopen(CFG.smet_id2mitre_url, timeout=10) as r:
        id2mitre = json.loads(r.read().decode("utf-8"))
    print(f"\nid2mitre.json fetched: {len(id2mitre)} entries")
    # id2mitre maps name→tcode or tcode→name — inspect first entry
    first_k, first_v = next(iter(id2mitre.items()))
    print(f"  Sample: {repr(first_k)} → {repr(first_v)}")
    # if values are T-codes, merge into name_to_tcode
    for k, v in id2mitre.items():
        kl = k.strip().lower()
        if isinstance(v, str) and re.match(r'T\d{4}', v):
            parent_tc = resolve_to_parent(v)
            if parent_tc in LABEL_INDEX:
                name_to_tcode[kl] = parent_tc
        elif isinstance(v, str):
            vl = v.strip().lower()
            if kl not in name_to_tcode and vl in name_to_tcode:
                name_to_tcode[kl] = name_to_tcode[vl]
    print(f"Name→T-code after id2mitre merge: {len(name_to_tcode)}")
except Exception as e:
    print(f"\nid2mitre fetch failed ({e}) — using STIX names only")

# ── 3. Parse SMET techniques column ──────────────────────────────
tech_col = [c for c in df_smet.columns
            if 'echnique' in c or 'ttack' in c][0]
print(f"\nUsing column: '{tech_col}'")

def parse_smet_techniques(val):
    """Parse string repr of list → list of T-codes in our label space."""
    if pd.isna(val):
        return []
    try:
        names = ast.literal_eval(str(val))
    except Exception:
        names = [str(val)]
    tcodes = []
    for name in names:
        nl = name.strip().lower()
        tc = name_to_tcode.get(nl)
        if tc:
            tcodes.append(tc)
    return tcodes

df_smet["tcodes"] = df_smet[tech_col].apply(parse_smet_techniques)

# Coverage stats
n_mapped   = (df_smet["tcodes"].apply(len) > 0).sum()
n_unmapped = len(df_smet) - n_mapped
all_smet_tcodes = set(t for ts in df_smet["tcodes"] for t in ts)

print(f"\nSMET rows with ≥1 mapped T-code : {n_mapped} / {len(df_smet)}")
print(f"Rows with no mapping            : {n_unmapped}")
print(f"Unique T-codes in SMET          : {len(all_smet_tcodes)}")
print(f"T-codes in our label space      : "
      f"{len(all_smet_tcodes & set(LABEL_INDEX))}")
print(f"T-codes outside our space       : "
      f"{sorted(all_smet_tcodes - set(LABEL_INDEX))}")

# Show unmapped technique names for diagnosis
unmapped_names = set()
for val in df_smet[tech_col]:
    try:
        names = ast.literal_eval(str(val))
        for n in names:
            if name_to_tcode.get(n.strip().lower()) is None:
                unmapped_names.add(n.strip())
    except:
        pass
if unmapped_names:
    print(f"\nUnmapped technique names ({len(unmapped_names)}):")
    for n in sorted(unmapped_names)[:20]:
        print(f"  '{n}'")

# ── 4. Filter to rows with ≥1 mapped technique ───────────────────
df_smet_eval = df_smet[df_smet["tcodes"].apply(len) > 0].copy()
df_smet_eval = df_smet_eval.reset_index(drop=True)
print(f"\nSMET eval rows (≥1 mapped tech) : {len(df_smet_eval)}")

# ── 5. Inference ──────────────────────────────────────────────────
print("Running inference on SMET...", flush=True)
model.eval()
smet_probs  = infer_probs(model, df_smet_eval["Description"].tolist())
smet_labels = mlb.transform(df_smet_eval["tcodes"].tolist()).astype(np.float32)
print(f"  smet_probs shape  : {smet_probs.shape}")
print(f"  smet_labels shape : {smet_labels.shape}")
print(f"  label density     : {smet_labels.mean():.4f}")

# ── 6. BRON-masked probs ──────────────────────────────────────────
# SMET has no CWE data — build dummy cwes column
df_smet_eval["cwes"] = [[] for _ in range(len(df_smet_eval))]
smet_probs_masked = apply_bron_mask_batch(
    smet_probs, df_smet_eval, LABEL_INDEX)

# ── 7. SMET-style metrics ─────────────────────────────────────────
def smet_metrics(probs, labels, k, tag):
    lrap = label_ranking_average_precision_score(labels, probs)
    rl   = label_ranking_loss(labels, probs)
    ce   = coverage_error(labels, probs)
    r5   = recall_at_k(labels, probs, k)
    print(f"\n── SMET {tag} ───────────────────────────────────────")
    print(f"  Coverage Error : {ce:.4f}   (SMET paper: 13.96)")
    print(f"  Ranking Loss   : {rl:.4f}   (SMET paper:  0.05)")
    print(f"  LRAP           : {lrap:.4f}  (SMET paper: 53.77%)")
    print(f"  R@{k}           : {r5:.4f}  (SMET paper: 67.71%)")
    return dict(coverage_error=ce, ranking_loss=rl, lrap=lrap, r5=r5)

r_smet_raw    = smet_metrics(smet_probs,        smet_labels, 5,
                              "NO mask")
r_smet_masked = smet_metrics(smet_probs_masked, smet_labels, 5,
                              "BRON mask")

# ── 8. Store results ──────────────────────────────────────────────
CFG._results["smet_raw"]    = r_smet_raw
CFG._results["smet_masked"] = r_smet_masked

print("\n✓ SMET evaluation done. Ready for Cell 14 (summary table).")

Name→T-code entries : 414
Sample:
  'screen capture' → T1113
  'boot or logon initialization scripts' → T1037
  'adversary-in-the-middle' → T1557
  'system owner/user discovery' → T1033
  'gather victim host information' → T1592

id2mitre.json fetched: 594 entries
  Sample: 'T1055.011' → 'Process Injection: Extra Window Memory Injection'
Name→T-code after id2mitre merge: 505

Using column: 'ATT&CK Techniques'

SMET rows with ≥1 mapped T-code : 168 / 303
Rows with no mapping            : 135
Unique T-codes in SMET          : 29
T-codes in our label space      : 29
T-codes outside our space       : []

Unmapped technique names (12):
  'Account Access Removal'
  'Command and Scripting Interpreter'
  'Create Account'
  'Data Destruction'
  'Drive-by Compromise'
  'Exploit Public-Facing Application'
  'Exploitation for Client Execution'
  'Exploitation for Privilege Escalation'
  'Indicator Removal on Host'
  'Software Discovery'
  'System Shutdown/Reboot'
  'User Execution'

SMET eval rows

In [22]:
# Cell 12 (continued) — KEV/CTID Full Evaluation

# ── 2. Group by CVE → set of parent techniques ───────────────────
kev_by_cve = defaultdict(set)   # cve_id → set of parent tcodes
kev_desc   = {}                  # cve_id → description (for tokenization)

for entry in kev_entries:
    cve_id = entry.get("capability_id", "").strip()
    tc_raw = entry.get("attack_object_id", "").strip()
    desc   = entry.get("capability_description", "").strip()
    if not cve_id.startswith("CVE-") or not tc_raw:
        continue
    # resolve sub→parent, revoked→current
    parent_tc = resolve_to_parent(tc_raw)
    if parent_tc in LABEL_INDEX:
        kev_by_cve[cve_id].add(parent_tc)
    if desc and cve_id not in kev_desc:
        kev_desc[cve_id] = desc

print(f"Unique CVEs in KEV              : {len(kev_by_cve)}")
print(f"CVEs with ≥1 label in our space : "
      f"{sum(1 for v in kev_by_cve.values() if v)}")

# Techniques not in our label space
all_kev_raw = {e.get('attack_object_id','') for e in kev_entries}
outside = {t for t in all_kev_raw
           if resolve_to_parent(t) not in LABEL_INDEX and t}
print(f"Unique T-codes in KEV           : {len(all_kev_raw)}")
print(f"T-codes outside our label space : {sorted(outside)[:10]}")

# ── 3. Build KEV dataframe ────────────────────────────────────────
kev_rows = []
for cve_id, techs in kev_by_cve.items():
    if not techs:
        continue
    # get CWEs from training data if available (for BRON mask)
    train_row = df_raw[df_raw["cve_id"] == cve_id]
    if len(train_row) > 0:
        cwes     = train_row.iloc[0]["cwes"]
        desc_use = train_row.iloc[0]["description"]   # enriched
        in_train = True
    else:
        cwes     = []
        desc_use = kev_desc.get(cve_id, "")
        in_train = False
    kev_rows.append({
        "cve_id"    : cve_id,
        "description": desc_use,
        "cwes"      : cwes,
        "techniques": list(techs),
        "in_train"  : in_train,
    })

df_kev = pd.DataFrame(kev_rows)
print(f"\nKEV eval rows                   : {len(df_kev)}")
print(f"  Overlap with training set     : {df_kev['in_train'].sum()}")
print(f"  Exclusive (not in train)      : {(~df_kev['in_train']).sum()}")

# ── 4. Tokenize & run inference ───────────────────────────────────
class SimpleDataset(Dataset):
    def __init__(self, descriptions, tokenizer, max_len):
        self.descriptions = descriptions
        self.tokenizer    = tokenizer
        self.max_len      = max_len
    def __len__(self):
        return len(self.descriptions)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.descriptions[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {"input_ids"     : enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0)}

@torch.no_grad()
def infer_probs(model, descriptions, batch_size=32):
    ds  = SimpleDataset(descriptions, tokenizer, CFG.max_len)
    dl  = DataLoader(ds, batch_size=batch_size, shuffle=False,
                     num_workers=0, pin_memory=False)
    all_logits = []
    for batch in dl:
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        with autocast():
            logits = model(ids, mask)
        all_logits.append(logits.float().cpu())
    logits_np = torch.cat(all_logits).numpy()
    return 1 / (1 + np.exp(-logits_np))

print("\nRunning inference on KEV set...", flush=True)
model.eval()
kev_probs = infer_probs(model, df_kev["description"].tolist())
kev_labels = mlb.transform(df_kev["techniques"].tolist()).astype(np.float32)
print(f"  kev_probs shape  : {kev_probs.shape}")
print(f"  kev_labels shape : {kev_labels.shape}")
print(f"  kev_labels density: {kev_labels.mean():.4f}")

# ── 5. BRON-masked probs ──────────────────────────────────────────
kev_probs_masked = apply_bron_mask_batch(kev_probs, df_kev, LABEL_INDEX)

# ── 6. Metrics function (KEV subset) ─────────────────────────────
def kev_metrics(probs, labels, threshold, tag):
    preds = (probs >= threshold).astype(int)
    mif1  = f1_score(labels, preds, average="micro",    zero_division=0)
    maf1  = f1_score(labels, preds, average="macro",    zero_division=0)
    lrap  = label_ranking_average_precision_score(labels, probs)
    r5    = recall_at_k(labels, probs, 5)
    print(f"\n── KEV {tag} (threshold={threshold:.2f}) ───────────────")
    print(f"  micro-F1 : {mif1:.4f}   macro-F1 : {maf1:.4f}")
    print(f"  LRAP     : {lrap:.4f}   R@5       : {r5:.4f}")
    return dict(micro_f1=mif1, macro_f1=maf1, lrap=lrap, r5=r5)

t = CFG._results["best_threshold"]

# Full KEV
r_kev_raw    = kev_metrics(kev_probs,        kev_labels, t,
                            "full — NO mask")
r_kev_masked = kev_metrics(kev_probs_masked, kev_labels, t,
                            "full — BRON mask")

# Exclusive only (not in training set)
excl_mask = ~df_kev["in_train"].values
if excl_mask.sum() > 0:
    r_kev_excl_raw    = kev_metrics(kev_probs[excl_mask],
                                    kev_labels[excl_mask], t,
                                    "exclusive — NO mask")
    r_kev_excl_masked = kev_metrics(kev_probs_masked[excl_mask],
                                    kev_labels[excl_mask], t,
                                    "exclusive — BRON mask")
else:
    print("  (no exclusive CVEs — all KEV overlap with training)")
    r_kev_excl_raw = r_kev_excl_masked = None

# ── 7. Store results ──────────────────────────────────────────────
CFG._results["kev_raw"]        = r_kev_raw
CFG._results["kev_masked"]     = r_kev_masked
CFG._results["kev_excl_raw"]   = r_kev_excl_raw
CFG._results["kev_excl_masked"]= r_kev_excl_masked

print("\n✓ KEV evaluation done. Ready for Cell 13.")

NameError: name 'kev_entries' is not defined

In [23]:
# Cell 12 (continued) — KEV/CTID Full Evaluation

# ── 2. Group by CVE → set of parent techniques ───────────────────
kev_by_cve = defaultdict(set)   # cve_id → set of parent tcodes
kev_desc   = {}                  # cve_id → description (for tokenization)

for entry in kev_entries:
    cve_id = entry.get("capability_id", "").strip()
    tc_raw = entry.get("attack_object_id", "").strip()
    desc   = entry.get("capability_description", "").strip()
    if not cve_id.startswith("CVE-") or not tc_raw:
        continue
    # resolve sub→parent, revoked→current
    parent_tc = resolve_to_parent(tc_raw)
    if parent_tc in LABEL_INDEX:
        kev_by_cve[cve_id].add(parent_tc)
    if desc and cve_id not in kev_desc:
        kev_desc[cve_id] = desc

print(f"Unique CVEs in KEV              : {len(kev_by_cve)}")
print(f"CVEs with ≥1 label in our space : "
      f"{sum(1 for v in kev_by_cve.values() if v)}")

# Techniques not in our label space
all_kev_raw = {e.get('attack_object_id','') for e in kev_entries}
outside = {t for t in all_kev_raw
           if resolve_to_parent(t) not in LABEL_INDEX and t}
print(f"Unique T-codes in KEV           : {len(all_kev_raw)}")
print(f"T-codes outside our label space : {sorted(outside)[:10]}")

# ── 3. Build KEV dataframe ────────────────────────────────────────
kev_rows = []
for cve_id, techs in kev_by_cve.items():
    if not techs:
        continue
    # get CWEs from training data if available (for BRON mask)
    train_row = df_raw[df_raw["cve_id"] == cve_id]
    if len(train_row) > 0:
        cwes     = train_row.iloc[0]["cwes"]
        desc_use = train_row.iloc[0]["description"]   # enriched
        in_train = True
    else:
        cwes     = []
        desc_use = kev_desc.get(cve_id, "")
        in_train = False
    kev_rows.append({
        "cve_id"    : cve_id,
        "description": desc_use,
        "cwes"      : cwes,
        "techniques": list(techs),
        "in_train"  : in_train,
    })

df_kev = pd.DataFrame(kev_rows)
print(f"\nKEV eval rows                   : {len(df_kev)}")
print(f"  Overlap with training set     : {df_kev['in_train'].sum()}")
print(f"  Exclusive (not in train)      : {(~df_kev['in_train']).sum()}")

# ── 4. Tokenize & run inference ───────────────────────────────────
class SimpleDataset(Dataset):
    def __init__(self, descriptions, tokenizer, max_len):
        self.descriptions = descriptions
        self.tokenizer    = tokenizer
        self.max_len      = max_len
    def __len__(self):
        return len(self.descriptions)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.descriptions[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {"input_ids"     : enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0)}

@torch.no_grad()
def infer_probs(model, descriptions, batch_size=32):
    ds  = SimpleDataset(descriptions, tokenizer, CFG.max_len)
    dl  = DataLoader(ds, batch_size=batch_size, shuffle=False,
                     num_workers=0, pin_memory=False)
    all_logits = []
    for batch in dl:
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        with autocast():
            logits = model(ids, mask)
        all_logits.append(logits.float().cpu())
    logits_np = torch.cat(all_logits).numpy()
    return 1 / (1 + np.exp(-logits_np))

print("\nRunning inference on KEV set...", flush=True)
model.eval()
kev_probs = infer_probs(model, df_kev["description"].tolist())
kev_labels = mlb.transform(df_kev["techniques"].tolist()).astype(np.float32)
print(f"  kev_probs shape  : {kev_probs.shape}")
print(f"  kev_labels shape : {kev_labels.shape}")
print(f"  kev_labels density: {kev_labels.mean():.4f}")

# ── 5. BRON-masked probs ──────────────────────────────────────────
kev_probs_masked = apply_bron_mask_batch(kev_probs, df_kev, LABEL_INDEX)

# ── 6. Metrics function (KEV subset) ─────────────────────────────
def kev_metrics(probs, labels, threshold, tag):
    preds = (probs >= threshold).astype(int)
    mif1  = f1_score(labels, preds, average="micro",    zero_division=0)
    maf1  = f1_score(labels, preds, average="macro",    zero_division=0)
    lrap  = label_ranking_average_precision_score(labels, probs)
    r5    = recall_at_k(labels, probs, 5)
    print(f"\n── KEV {tag} (threshold={threshold:.2f}) ───────────────")
    print(f"  micro-F1 : {mif1:.4f}   macro-F1 : {maf1:.4f}")
    print(f"  LRAP     : {lrap:.4f}   R@5       : {r5:.4f}")
    return dict(micro_f1=mif1, macro_f1=maf1, lrap=lrap, r5=r5)

t = CFG._results["best_threshold"]

# Full KEV
r_kev_raw    = kev_metrics(kev_probs,        kev_labels, t,
                            "full — NO mask")
r_kev_masked = kev_metrics(kev_probs_masked, kev_labels, t,
                            "full — BRON mask")

# Exclusive only (not in training set)
excl_mask = ~df_kev["in_train"].values
if excl_mask.sum() > 0:
    r_kev_excl_raw    = kev_metrics(kev_probs[excl_mask],
                                    kev_labels[excl_mask], t,
                                    "exclusive — NO mask")
    r_kev_excl_masked = kev_metrics(kev_probs_masked[excl_mask],
                                    kev_labels[excl_mask], t,
                                    "exclusive — BRON mask")
else:
    print("  (no exclusive CVEs — all KEV overlap with training)")
    r_kev_excl_raw = r_kev_excl_masked = None

# ── 7. Store results ──────────────────────────────────────────────
CFG._results["kev_raw"]        = r_kev_raw
CFG._results["kev_masked"]     = r_kev_masked
CFG._results["kev_excl_raw"]   = r_kev_excl_raw
CFG._results["kev_excl_masked"]= r_kev_excl_masked

print("\n✓ KEV evaluation done. Ready for Cell 13.")

NameError: name 'kev_entries' is not defined

In [35]:
# Cell 12-fix-v2 — KEV Evaluation with direct NVD re-parse

# ── 1. Re-parse NVD files for ALL CVEs (not just training subset) ─
import time

def parse_cvss_string(impact):
    """Extract CVSS vector components from NVD impact field."""
    cvss = None
    if "baseMetricV3" in impact:
        cvss = impact["baseMetricV3"].get("cvssV3", {})
    elif "baseMetricV2" in impact:
        cvss = impact["baseMetricV2"].get("cvssV2", {})
    if not cvss:
        return ""
    
    parts = []
    mapping = {
        "attackVector": "AV", "accessVector": "AV",
        "attackComplexity": "AC", "accessComplexity": "AC",
        "privilegesRequired": "PR",
        "userInteraction": "UI",
        "scope": "S",
        "confidentialityImpact": "C",
        "integrityImpact": "I",
        "availabilityImpact": "A",
    }
    for json_key, short in mapping.items():
        val = cvss.get(json_key, "")
        if val:
            parts.append(f"{short}:{val.upper()}")
    
    score = cvss.get("baseScore", "")
    if score:
        parts.append(f"SCORE:{score}")
    
    return "[" + " ".join(parts) + "]" if parts else ""


def enrich_description(desc, cwes, impact):
    """Prepend CVSS + CWE tags to description."""
    prefix_parts = []
    cvss_str = parse_cvss_string(impact)
    if cvss_str:
        prefix_parts.append(cvss_str)
    if cwes:
        prefix_parts.append("[CWE: " + ", ".join(cwes) + "]")
    if prefix_parts:
        return " ".join(prefix_parts) + " " + desc
    return desc


# Build a full CVE lookup from ALL NVD files (description + CWEs + CVSS)
print("Re-parsing NVD files for full CVE lookup...", flush=True)
t0 = time.time()

nvd_full_lookup = {}  # cve_id → {"description": ..., "cwes": [...]}
nvd_files = sorted(glob.glob("nvdcve-1.1-*.json"))

for fpath in nvd_files:
    with open(fpath, encoding="utf-8") as f:
        data = json.load(f)
    for item in data.get("CVE_Items", []):
        cve_id = item["cve"]["CVE_data_meta"]["ID"]
        
        # Description
        desc_data = item["cve"]["description"]["description_data"]
        desc = next((d["value"] for d in desc_data
                     if d.get("lang", "en") == "en"), "")
        if not desc:
            desc = desc_data[0]["value"] if desc_data else ""
        
        # CWEs
        cwes = []
        for pt in item["cve"].get("problemtype", {}).get("problemtype_data", []):
            for d in pt.get("description", []):
                val = d.get("value", "")
                if val.startswith("CWE-") and val not in (
                    "NVD-CWE-Other", "NVD-CWE-noinfo"):
                    cwes.append(val)
        
        # Impact / CVSS
        impact = item.get("impact", {})
        
        # Build enriched description
        enriched = enrich_description(desc, cwes, impact)
        
        nvd_full_lookup[cve_id] = {
            "description": enriched,
            "cwes": cwes,
            "raw_desc": desc,
        }

print(f"  Parsed {len(nvd_full_lookup):,} CVEs in {time.time()-t0:.1f}s")

# ── 2. Rebuild KEV dataframe with full NVD descriptions ──────────
kev_by_cve = defaultdict(set)
for entry in kev_entries:
    cve_id = entry.get("capability_id", "").strip()
    tc_raw = entry.get("attack_object_id", "").strip()
    if not cve_id.startswith("CVE-") or not tc_raw:
        continue
    parent_tc = resolve_to_parent(tc_raw)
    if parent_tc in LABEL_INDEX:
        kev_by_cve[cve_id].add(parent_tc)

train_cve_set = set(df_raw["cve_id"])

kev_rows = []
source_stats = Counter()

for cve_id, techs in kev_by_cve.items():
    if not techs:
        continue
    
    in_train = cve_id in train_cve_set
    
    if cve_id in nvd_full_lookup:
        info = nvd_full_lookup[cve_id]
        desc_use = info["description"]
        cwes = info["cwes"]
        source_stats["nvd_enriched"] += 1
    else:
        # Fallback to KEV short description
        kev_entry = next((e for e in kev_entries
                          if e.get("capability_id") == cve_id), None)
        desc_use = kev_entry.get("capability_description", "") if kev_entry else ""
        cwes = []
        source_stats["kev_fallback"] += 1
    
    if not desc_use:
        source_stats["skipped"] += 1
        continue
    
    kev_rows.append({
        "cve_id"     : cve_id,
        "description": desc_use,
        "cwes"       : cwes,
        "techniques" : list(techs),
        "in_train"   : in_train,
    })

df_kev = pd.DataFrame(kev_rows)
print(f"\nKEV eval rows              : {len(df_kev)}")
print(f"  Source stats             : {dict(source_stats)}")
print(f"  Enriched (has [prefix)   : {df_kev['description'].str.startswith('[').sum()}")
print(f"  Has CWEs                 : {(df_kev['cwes'].apply(len) > 0).sum()}")
print(f"  Overlap with training    : {df_kev['in_train'].sum()}")
print(f"  Exclusive (not in train) : {(~df_kev['in_train']).sum()}")

# Show samples
print(f"\n── Sample exclusive CVEs ───────────────────────────────")
excl = df_kev[~df_kev["in_train"]]
for _, row in excl.head(5).iterrows():
    print(f"  [{row['cve_id']}] len={len(row['description'])} "
          f"cwes={row['cwes'][:2]}")
    print(f"    {row['description'][:150]}...")

# ── 3. Inference — all 4 modes ───────────────────────────────────
print("\n── Running KEV inference ───────────────────────────────")
model.eval()
kev_probs = infer_probs(model, df_kev["description"].tolist())
kev_labels = mlb.transform(df_kev["techniques"].tolist()).astype(np.float32)
print(f"  probs: {kev_probs.shape}  labels: {kev_labels.shape}")

kev_probs_masked = apply_bron_mask_batch(kev_probs, df_kev, LABEL_INDEX)

print("  Sentence max-pool...")
kev_probs_smp = sentence_maxpool_inference(
    model, df_kev["description"].tolist(), batch_size=64)
kev_probs_smp_masked = apply_bron_mask_batch(
    kev_probs_smp, df_kev, LABEL_INDEX)

# ── 4. Threshold sweep ───────────────────────────────────────────
print("\n── KEV threshold sweep (full-text, no mask) ────────────")
print(f"  {'Thresh':>7}  {'MiF1':>7}  {'MaF1':>7}  {'LRAP':>7}  {'R@5':>7}")
best_kev_mif1 = -1
best_kev_thresh = 0.5
for thr in np.arange(0.15, 0.65, 0.05):
    preds = (kev_probs >= thr).astype(int)
    mif1 = f1_score(kev_labels, preds, average="micro", zero_division=0)
    maf1 = f1_score(kev_labels, preds, average="macro", zero_division=0)
    lrap = label_ranking_average_precision_score(kev_labels, kev_probs)
    r5   = recall_at_k(kev_labels, kev_probs, 5)
    print(f"  {thr:>7.2f}  {mif1:>7.4f}  {maf1:>7.4f}  {lrap:>7.4f}  {r5:>7.4f}")
    if mif1 > best_kev_mif1:
        best_kev_mif1 = mif1
        best_kev_thresh = thr

print(f"\n  Best KEV threshold: {best_kev_thresh:.2f} (micro-F1={best_kev_mif1:.4f})")

# ── 5. Full metrics — all 4 modes ────────────────────────────────
t = best_kev_thresh

print(f"\n{'='*60}")
print(f"  KEV FULL SET — threshold={t:.2f}")
print(f"{'='*60}")
r_kev_raw    = kev_metrics(kev_probs,            kev_labels, t, "Full-text, NO mask")
r_kev_masked = kev_metrics(kev_probs_masked,     kev_labels, t, "Full-text, BRON mask")
r_kev_smp    = kev_metrics(kev_probs_smp,        kev_labels, t, "SMP, NO mask")
r_kev_smp_m  = kev_metrics(kev_probs_smp_masked, kev_labels, t, "SMP, BRON mask")

excl_mask = ~df_kev["in_train"].values
if excl_mask.sum() > 0:
    print(f"\n{'='*60}")
    print(f"  KEV EXCLUSIVE ({excl_mask.sum()} CVEs) — threshold={t:.2f}")
    print(f"{'='*60}")
    r_excl_raw   = kev_metrics(kev_probs[excl_mask],            kev_labels[excl_mask], t, "Full-text, NO mask")
    r_excl_mask  = kev_metrics(kev_probs_masked[excl_mask],     kev_labels[excl_mask], t, "Full-text, BRON mask")
    r_excl_smp   = kev_metrics(kev_probs_smp[excl_mask],        kev_labels[excl_mask], t, "SMP, NO mask")
    r_excl_smp_m = kev_metrics(kev_probs_smp_masked[excl_mask], kev_labels[excl_mask], t, "SMP, BRON mask")

# ── 6. Spot check ────────────────────────────────────────────────
print(f"\n── Spot check (enriched exclusive CVEs) ────────────────")
enriched_excl = df_kev[(~df_kev["in_train"]) & 
                        (df_kev["description"].str.startswith("["))]
for _, row in enriched_excl.head(5).iterrows():
    i = df_kev.index.get_loc(row.name)
    probs_i = kev_probs[i]
    top5_idx = np.argsort(-probs_i)[:5]
    top5 = [(LABEL_LIST[j], f"{probs_i[j]:.3f}") for j in top5_idx]
    gold = row["techniques"]
    gold_in_top5 = sum(1 for g in gold if g in [LABEL_LIST[j] for j in top5_idx])
    print(f"\n  [{row['cve_id']}] len={len(row['description'])} cwes={row['cwes'][:2]}")
    print(f"    Gold : {gold}  (hits in top5: {gold_in_top5})")
    print(f"    Top-5: {top5}")

# ── 7. Store ──────────────────────────────────────────────────────
CFG._results["kev_raw"]         = r_kev_raw
CFG._results["kev_masked"]      = r_kev_masked
CFG._results["kev_smp_raw"]     = r_kev_smp
CFG._results["kev_smp_masked"]  = r_kev_smp_m
CFG._results["kev_best_threshold"] = best_kev_thresh
if excl_mask.sum() > 0:
    CFG._results["kev_excl_raw"]        = r_excl_raw
    CFG._results["kev_excl_masked"]     = r_excl_mask
    CFG._results["kev_excl_smp_raw"]    = r_excl_smp
    CFG._results["kev_excl_smp_masked"] = r_excl_smp_m

print(f"\n✓ KEV evaluation complete. Ready for Cell 13 (SMET).")

Re-parsing NVD files for full CVE lookup...


  Parsed 0 CVEs in 0.0s

KEV eval rows              : 248
  Source stats             : {'kev_fallback': 248}
  Enriched (has [prefix)   : 0
  Has CWEs                 : 0
  Overlap with training    : 75
  Exclusive (not in train) : 173

── Sample exclusive CVEs ───────────────────────────────
  [CVE-2024-34102] len=112 cwes=[]
    Adobe Commerce and Magento Open Source Improper Restriction of XML External Entity Reference (XXE) Vulnerability...
  [CVE-2019-13608] len=75 cwes=[]
    Citrix StoreFront Server XML External Entity (XXE) Processing Vulnerability...
  [CVE-2022-3038] len=60 cwes=[]
    Google Chromium Network Service Use-After-Free Vulnerability...
  [CVE-2021-29256] len=55 cwes=[]
    Arm Mali GPU Kernel Driver Use-After-Free Vulnerability...
  [CVE-2015-5119] len=47 cwes=[]
    Adobe Flash Player Use-After-Free Vulnerability...

── Running KEV inference ───────────────────────────────
  probs: (248, 96)  labels: (248, 96)
  Sentence max-pool...
    Total sentences to proces

In [26]:
# Cell 13 — SMET 303 Benchmark Evaluation

# ── 1. Load SMET dataset ─────────────────────────────────────────
df_smet = pd.read_excel("CVE_annotated_dataset.xlsx")
print(f"SMET dataset shape: {df_smet.shape}")
print(f"Columns: {list(df_smet.columns)}")
print(f"First row sample:")
print(f"  ID: {df_smet.iloc[0]['ID']}")
print(f"  Description: {df_smet.iloc[0]['Description'][:150]}...")
print(f"  ATT&CK Techniques: {df_smet.iloc[0]['ATT&CK Techniques']}")

# ── 2. Build name→T-code mapping from ATT&CK STIX ────────────────
with open("OSRs/ATTACK/enterprise-attack-v16.1.json", encoding="utf-8") as f:
    stix = json.load(f)

name_to_tcode = {}
tcode_to_name = {}
for obj in stix["objects"]:
    if obj.get("type") != "attack-pattern":
        continue
    if obj.get("revoked", False) or obj.get("x_mitre_deprecated", False):
        continue
    name = obj.get("name", "")
    ext_refs = obj.get("external_references", [])
    for ref in ext_refs:
        if ref.get("source_name") == "mitre-attack":
            tc = ref.get("external_id", "")
            if tc:
                name_to_tcode[name.lower().strip()] = tc
                tcode_to_name[tc] = name
                break

print(f"\nATT&CK name→tcode mappings: {len(name_to_tcode)}")

# Also load id2mitre.json as fallback
id2mitre = {}
try:
    with open("id2mitre.json", encoding="utf-8") as f:
        id2mitre = json.load(f)
    print(f"id2mitre.json loaded: {len(id2mitre)} entries")
    # id2mitre maps string index → technique name; build name→tcode from it
    # Actually check its structure first
    sample_key = list(id2mitre.keys())[0]
    sample_val = id2mitre[sample_key]
    print(f"  Sample: '{sample_key}' → '{sample_val}'")
except Exception as e:
    print(f"id2mitre.json not available: {e}")

# ── 3. Parse SMET technique names → T-codes ──────────────────────
def parse_smet_techniques(tech_str):
    """Parse SMET 'ATT&CK Techniques' column → list of parent T-codes."""
    if pd.isna(tech_str) or not tech_str.strip():
        return []
    
    # Could be a string repr of a list like "['Technique1', 'Technique2']"
    try:
        tech_list = literal_eval(tech_str)
    except:
        # Try splitting by comma
        tech_list = [t.strip().strip("'\"") for t in tech_str.split(",")]
    
    if isinstance(tech_list, str):
        tech_list = [tech_list]
    
    tcodes = []
    for name in tech_list:
        name_clean = name.strip().strip("'\"")
        if not name_clean:
            continue
        
        # Direct lookup
        tc = name_to_tcode.get(name_clean.lower().strip())
        
        # Try id2mitre reverse lookup if direct fails
        if tc is None and id2mitre:
            for k, v in id2mitre.items():
                if v.lower().strip() == name_clean.lower().strip():
                    # k is an index, we need the tcode
                    # Check if v maps to a tcode
                    tc = name_to_tcode.get(v.lower().strip())
                    break
        
        if tc:
            # Resolve to parent
            parent = resolve_to_parent(tc)
            if parent in LABEL_INDEX:
                tcodes.append(parent)
        # else: technique not in our label space
    
    return list(set(tcodes))

# Apply parsing
df_smet["parsed_techniques"] = df_smet["ATT&CK Techniques"].apply(parse_smet_techniques)

# Stats
smet_all_names = set()
for tech_str in df_smet["ATT&CK Techniques"]:
    try:
        names = literal_eval(str(tech_str))
        if isinstance(names, list):
            smet_all_names.update(n.strip() for n in names)
    except:
        pass

mapped_names = set()
unmapped_names = set()
for name in smet_all_names:
    if name_to_tcode.get(name.lower().strip()):
        mapped_names.add(name)
    else:
        unmapped_names.add(name)

print(f"\nSMET unique technique names : {len(smet_all_names)}")
print(f"  Mapped to T-codes        : {len(mapped_names)}")
print(f"  Unmapped                 : {len(unmapped_names)}")
if unmapped_names:
    print(f"  Unmapped names: {sorted(unmapped_names)[:15]}")

# How many SMET techniques land in our label space?
smet_tcodes_in_space = set()
smet_tcodes_outside = set()
for techs in df_smet["parsed_techniques"]:
    for tc in techs:
        if tc in LABEL_INDEX:
            smet_tcodes_in_space.add(tc)
        else:
            smet_tcodes_outside.add(tc)

print(f"\nSMET T-codes in our label space : {len(smet_tcodes_in_space)}")
print(f"SMET T-codes outside our space  : {len(smet_tcodes_outside)}")
if smet_tcodes_outside:
    print(f"  Outside: {sorted(smet_tcodes_outside)[:10]}")

# Filter to rows with at least 1 technique in our space
df_smet_eval = df_smet[df_smet["parsed_techniques"].apply(len) > 0].copy()
print(f"\nSMET rows with ≥1 technique in our space: {len(df_smet_eval)} / {len(df_smet)}")

# Check overlap with training
smet_cve_ids = set(df_smet_eval["ID"])
train_cve_ids = set(df_raw["cve_id"])
overlap = smet_cve_ids & train_cve_ids
print(f"SMET CVEs overlapping with training: {len(overlap)}")

# ── 4. Enrich SMET descriptions with CVSS/CWE from NVD ───────────
# Use nvd_full_lookup if available, else df_raw lookup
smet_descriptions = []
smet_cwes_list = []
for _, row in df_smet_eval.iterrows():
    cve_id = row["ID"]
    desc = row["Description"]
    
    # Try to get enriched version from training data
    if cve_id in nvd_lookup:
        desc = nvd_lookup[cve_id]["description"]
        cwes = nvd_lookup[cve_id]["cwes"]
    else:
        cwes = []
    
    smet_descriptions.append(desc)
    smet_cwes_list.append(cwes)

df_smet_eval = df_smet_eval.copy()
df_smet_eval["enriched_desc"] = smet_descriptions
df_smet_eval["cwes"] = smet_cwes_list

enriched_count = sum(1 for d in smet_descriptions if d.startswith("["))
print(f"SMET descriptions enriched: {enriched_count} / {len(df_smet_eval)}")

# ── 5. Inference — all 4 modes ───────────────────────────────────
print("\n── Running SMET inference ──────────────────────────────")
model.eval()

# Full-text
smet_probs = infer_probs(model, df_smet_eval["enriched_desc"].tolist())
smet_labels = mlb.transform(
    df_smet_eval["parsed_techniques"].tolist()).astype(np.float32)
print(f"  probs: {smet_probs.shape}  labels: {smet_labels.shape}")
print(f"  label density: {smet_labels.mean():.4f}")

# BRON masked
smet_probs_masked = apply_bron_mask_batch(
    smet_probs, df_smet_eval, LABEL_INDEX)

# Sentence max-pool
print("  Sentence max-pool...")
smet_probs_smp = sentence_maxpool_inference(
    model, df_smet_eval["enriched_desc"].tolist(), batch_size=64)
smet_probs_smp_masked = apply_bron_mask_batch(
    smet_probs_smp, df_smet_eval, LABEL_INDEX)

# ── 6. SMET-style metrics ────────────────────────────────────────
def smet_metrics(probs, labels, threshold, tag):
    """Compute SMET paper metrics: Coverage Error, Ranking Loss, LRAP, R@5."""
    preds = (probs >= threshold).astype(int)
    
    ce   = coverage_error(labels, probs)
    rl   = label_ranking_loss(labels, probs)
    lrap = label_ranking_average_precision_score(labels, probs)
    r5   = recall_at_k(labels, probs, 5)
    mif1 = f1_score(labels, preds, average="micro", zero_division=0)
    maf1 = f1_score(labels, preds, average="macro", zero_division=0)
    
    print(f"\n── SMET {tag} (threshold={threshold:.2f}) ─────────────")
    print(f"  Coverage Error : {ce:.2f}")
    print(f"  Ranking Loss   : {rl:.4f}")
    print(f"  LRAP           : {lrap*100:.2f}%")
    print(f"  R@5            : {r5*100:.2f}%")
    print(f"  micro-F1       : {mif1:.4f}")
    print(f"  macro-F1       : {maf1:.4f}")
    
    return dict(coverage_error=ce, ranking_loss=rl, lrap=lrap,
                lrap_pct=lrap*100, r5=r5, r5_pct=r5*100,
                micro_f1=mif1, macro_f1=maf1, threshold=threshold)

# Threshold sweep for SMET
print("\n── SMET threshold sweep (full-text, no mask) ───────────")
print(f"  {'Thresh':>7}  {'MiF1':>7}  {'CovErr':>7}  {'RkLoss':>7}  {'LRAP%':>7}  {'R@5%':>7}")
best_smet_mif1 = -1
best_smet_thresh = 0.5
for thr in np.arange(0.15, 0.65, 0.05):
    preds = (smet_probs >= thr).astype(int)
    mif1 = f1_score(smet_labels, preds, average="micro", zero_division=0)
    ce   = coverage_error(smet_labels, smet_probs)
    rl   = label_ranking_loss(smet_labels, smet_probs)
    lrap = label_ranking_average_precision_score(smet_labels, smet_probs)
    r5   = recall_at_k(smet_labels, smet_probs, 5)
    print(f"  {thr:>7.2f}  {mif1:>7.4f}  {ce:>7.2f}  {rl:>7.4f}  {lrap*100:>7.2f}  {r5*100:>7.2f}")
    if mif1 > best_smet_mif1:
        best_smet_mif1 = mif1
        best_smet_thresh = thr

print(f"\n  Best SMET threshold: {best_smet_thresh:.2f}")

# All 4 modes
t = best_smet_thresh
print(f"\n{'='*60}")
print(f"  SMET 303 BENCHMARK — threshold={t:.2f}")
print(f"{'='*60}")

r_smet_raw    = smet_metrics(smet_probs,            smet_labels, t, "Full-text, NO mask")
r_smet_masked = smet_metrics(smet_probs_masked,     smet_labels, t, "Full-text, BRON mask")
r_smet_smp    = smet_metrics(smet_probs_smp,        smet_labels, t, "SMP, NO mask")
r_smet_smp_m  = smet_metrics(smet_probs_smp_masked, smet_labels, t, "SMP, BRON mask")

# ── 7. Comparison with SMET published numbers ────────────────────
print(f"\n{'='*60}")
print(f"  COMPARISON WITH SMET PAPER")
print(f"{'='*60}")
print(f"  {'Metric':<18} {'SMET Paper':>12} {'Ours (best)':>12} {'Delta':>10}")
print(f"  {'─'*18} {'─'*12} {'─'*12} {'─'*10}")

smet_paper = {"Coverage Error": 13.96, "Ranking Loss": 0.05,
              "LRAP%": 53.77, "R@5%": 67.71}

# Find best mode for each metric
all_modes = {
    "Full-text":     r_smet_raw,
    "Full+BRON":     r_smet_masked,
    "SMP":           r_smet_smp,
    "SMP+BRON":      r_smet_smp_m,
}

# Best coverage error (lower is better)
best_ce = min(all_modes.values(), key=lambda x: x["coverage_error"])
# Best ranking loss (lower is better)
best_rl = min(all_modes.values(), key=lambda x: x["ranking_loss"])
# Best LRAP (higher is better)
best_lrap = max(all_modes.values(), key=lambda x: x["lrap_pct"])
# Best R@5 (higher is better)
best_r5 = max(all_modes.values(), key=lambda x: x["r5_pct"])

print(f"  {'Coverage Error':<18} {smet_paper['Coverage Error']:>12.2f} {best_ce['coverage_error']:>12.2f} {best_ce['coverage_error']-smet_paper['Coverage Error']:>+10.2f}")
print(f"  {'Ranking Loss':<18} {smet_paper['Ranking Loss']:>12.4f} {best_rl['ranking_loss']:>12.4f} {best_rl['ranking_loss']-smet_paper['Ranking Loss']:>+10.4f}")
print(f"  {'LRAP%':<18} {smet_paper['LRAP%']:>12.2f} {best_lrap['lrap_pct']:>12.2f} {best_lrap['lrap_pct']-smet_paper['LRAP%']:>+10.2f}")
print(f"  {'R@5%':<18} {smet_paper['R@5%']:>12.2f} {best_r5['r5_pct']:>12.2f} {best_r5['r5_pct']-smet_paper['R@5%']:>+10.2f}")

# ── 8. Store results ──────────────────────────────────────────────
CFG._results["smet_raw"]        = r_smet_raw
CFG._results["smet_masked"]     = r_smet_masked
CFG._results["smet_smp_raw"]    = r_smet_smp
CFG._results["smet_smp_masked"] = r_smet_smp_m
CFG._results["smet_best_threshold"] = best_smet_thresh

print(f"\n✓ SMET evaluation complete. Ready for Cell 14 (final summary).")

SMET dataset shape: (303, 4)
Columns: ['ID', 'Description', 'ATT&CK Techniques', 'Manually extracted attack vectors']
First row sample:
  ID: CVE-2021-29665
  Description: IBM Security Verify Access 20.07 is vulnerable to a stack based buffer overflow, caused by improper bounds checking which could allow a local attacker...
  ATT&CK Techniques: ['Exploitation for Privilege Escalation']

ATT&CK name→tcode mappings: 631
id2mitre.json loaded: 594 entries
  Sample: 'T1055.011' → 'Process Injection: Extra Window Memory Injection'

SMET unique technique names : 0
  Mapped to T-codes        : 0
  Unmapped                 : 0

SMET T-codes in our label space : 5
SMET T-codes outside our space  : 0

SMET rows with ≥1 technique in our space: 7 / 303
SMET CVEs overlapping with training: 0


NameError: name 'nvd_lookup' is not defined

In [38]:
# Cell 13-fix — SMET 303 Benchmark (Fixed parsing)

from ast import literal_eval

# ── 1. Parse technique names → T-codes ───────────────────────────
def parse_smet_techniques(tech_str):
    """Parse SMET 'ATT&CK Techniques' column → list of parent T-codes."""
    if pd.isna(tech_str) or not str(tech_str).strip():
        return []
    
    try:
        tech_list = literal_eval(str(tech_str))
    except:
        tech_list = [t.strip().strip("'\"[]") for t in str(tech_str).split(",")]
    
    if isinstance(tech_list, str):
        tech_list = [tech_list]
    
    tcodes = []
    for name in tech_list:
        name_clean = name.strip()
        if not name_clean:
            continue
        tc = name_to_tcode.get(name_clean.lower())
        if tc:
            parent = resolve_to_parent(tc)
            if parent in LABEL_INDEX:
                tcodes.append(parent)
    
    return list(set(tcodes))

# Apply
df_smet["parsed_techniques"] = df_smet["ATT&CK Techniques"].apply(parse_smet_techniques)

# ── 2. Coverage stats ────────────────────────────────────────────
all_smet_names = set()
all_smet_tcodes = set()
all_smet_parents = set()
unmapped_names = set()

for tech_str in df_smet["ATT&CK Techniques"]:
    try:
        names = literal_eval(str(tech_str))
    except:
        names = [t.strip().strip("'\"[]") for t in str(tech_str).split(",")]
    if isinstance(names, str):
        names = [names]
    for name in names:
        name = name.strip()
        all_smet_names.add(name)
        tc = name_to_tcode.get(name.lower())
        if tc:
            all_smet_tcodes.add(tc)
            parent = resolve_to_parent(tc)
            all_smet_parents.add(parent)
        else:
            unmapped_names.add(name)

in_space = all_smet_parents & set(LABEL_INDEX.keys())
out_space = all_smet_parents - set(LABEL_INDEX.keys())

print(f"SMET unique technique names  : {len(all_smet_names)}")
print(f"  Mapped to T-codes          : {len(all_smet_names) - len(unmapped_names)}")
print(f"  Unmapped names             : {len(unmapped_names)}")
if unmapped_names:
    print(f"    {sorted(unmapped_names)[:20]}")
print(f"SMET T-codes (raw)           : {len(all_smet_tcodes)}")
print(f"SMET T-codes (parent)        : {len(all_smet_parents)}")
print(f"  In our 96-label space      : {len(in_space)}  {sorted(in_space)[:15]}")
print(f"  Outside our space          : {len(out_space)}  {sorted(out_space)[:15]}")

# Filter
df_smet_eval = df_smet[df_smet["parsed_techniques"].apply(len) > 0].copy()
print(f"\nSMET rows with ≥1 in our space: {len(df_smet_eval)} / {len(df_smet)}")

# Overlap with training
smet_cve_ids = set(df_smet_eval["ID"])
train_cve_ids = set(df_raw["cve_id"])
overlap = smet_cve_ids & train_cve_ids
print(f"Overlap with training: {len(overlap)}")

# ── 3. Enrich descriptions ───────────────────────────────────────
nvd_lookup_map = {row["cve_id"]: {"description": row["description"], 
                   "cwes": row["cwes"]} for _, row in df_raw.iterrows()}

smet_descs = []
smet_cwes = []
for _, row in df_smet_eval.iterrows():
    cve_id = row["ID"]
    if cve_id in nvd_lookup_map:
        smet_descs.append(nvd_lookup_map[cve_id]["description"])
        smet_cwes.append(nvd_lookup_map[cve_id]["cwes"])
    else:
        smet_descs.append(row["Description"])
        smet_cwes.append([])

df_smet_eval["enriched_desc"] = smet_descs
df_smet_eval["cwes"] = smet_cwes

enriched_count = sum(1 for d in smet_descs if d.startswith("["))
print(f"Descriptions enriched (CVSS/CWE prefix): {enriched_count} / {len(df_smet_eval)}")

# ── 4. Inference ──────────────────────────────────────────────────
print("\n── Running SMET inference ──────────────────────────────")
model.eval()

smet_probs = infer_probs(model, df_smet_eval["enriched_desc"].tolist())
smet_labels = mlb.transform(
    df_smet_eval["parsed_techniques"].tolist()).astype(np.float32)
print(f"  probs: {smet_probs.shape}  labels: {smet_labels.shape}")
print(f"  label density: {smet_labels.mean():.4f}  "
      f"avg labels/row: {smet_labels.sum(1).mean():.2f}")

smet_probs_masked = apply_bron_mask_batch(smet_probs, df_smet_eval, LABEL_INDEX)

print("  Sentence max-pool...")
smet_probs_smp = sentence_maxpool_inference(
    model, df_smet_eval["enriched_desc"].tolist(), batch_size=64)
smet_probs_smp_masked = apply_bron_mask_batch(
    smet_probs_smp, df_smet_eval, LABEL_INDEX)

# ── 5. Threshold sweep ───────────────────────────────────────────
print(f"\n── SMET threshold sweep (full-text, no mask) ───────────")
print(f"  {'Thresh':>7}  {'MiF1':>7}  {'CovErr':>7}  {'RkLoss':>7}  {'LRAP%':>7}  {'R@5%':>7}")
best_smet_mif1 = -1
best_smet_thresh = 0.5
for thr in np.arange(0.10, 0.65, 0.05):
    preds = (smet_probs >= thr).astype(int)
    mif1 = f1_score(smet_labels, preds, average="micro", zero_division=0)
    ce   = coverage_error(smet_labels, smet_probs)
    rl   = label_ranking_loss(smet_labels, smet_probs)
    lrap = label_ranking_average_precision_score(smet_labels, smet_probs)
    r5   = recall_at_k(smet_labels, smet_probs, 5)
    print(f"  {thr:>7.2f}  {mif1:>7.4f}  {ce:>7.2f}  {rl:>7.4f}  {lrap*100:>7.2f}  {r5*100:>7.2f}")
    if mif1 > best_smet_mif1:
        best_smet_mif1 = mif1
        best_smet_thresh = thr

print(f"\n  Best SMET threshold: {best_smet_thresh:.2f}")

# ── 6. Full metrics — all 4 modes ────────────────────────────────
def smet_metrics(probs, labels, threshold, tag):
    preds = (probs >= threshold).astype(int)
    ce   = coverage_error(labels, probs)
    rl   = label_ranking_loss(labels, probs)
    lrap = label_ranking_average_precision_score(labels, probs)
    r5   = recall_at_k(labels, probs, 5)
    mif1 = f1_score(labels, preds, average="micro", zero_division=0)
    maf1 = f1_score(labels, preds, average="macro", zero_division=0)
    
    print(f"\n── SMET {tag} (threshold={threshold:.2f}) ─────────────")
    print(f"  Coverage Error : {ce:.2f}")
    print(f"  Ranking Loss   : {rl:.4f}")
    print(f"  LRAP           : {lrap*100:.2f}%")
    print(f"  R@5            : {r5*100:.2f}%")
    print(f"  micro-F1       : {mif1:.4f}")
    print(f"  macro-F1       : {maf1:.4f}")
    
    return dict(coverage_error=ce, ranking_loss=rl, lrap=lrap,
                lrap_pct=lrap*100, r5=r5, r5_pct=r5*100,
                micro_f1=mif1, macro_f1=maf1, threshold=threshold)

t = best_smet_thresh
print(f"\n{'='*60}")
print(f"  SMET BENCHMARK — threshold={t:.2f}")
print(f"{'='*60}")

r_smet_raw    = smet_metrics(smet_probs,            smet_labels, t, "Full-text, NO mask")
r_smet_masked = smet_metrics(smet_probs_masked,     smet_labels, t, "Full-text, BRON mask")
r_smet_smp    = smet_metrics(smet_probs_smp,        smet_labels, t, "SMP, NO mask")
r_smet_smp_m  = smet_metrics(smet_probs_smp_masked, smet_labels, t, "SMP, BRON mask")

# ── 7. Spot check ────────────────────────────────────────────────
print(f"\n── Spot check: first 10 SMET CVEs ─────────────────────")
for idx, (_, row) in enumerate(df_smet_eval.head(10).iterrows()):
    i = df_smet_eval.index.get_loc(row.name)
    probs_i = smet_probs[i]
    top5_idx = np.argsort(-probs_i)[:5]
    top5 = [(LABEL_LIST[j], f"{probs_i[j]:.3f}") for j in top5_idx]
    gold = row["parsed_techniques"]
    hits = sum(1 for g in gold if g in [LABEL_LIST[j] for j in top5_idx])
    print(f"\n  [{row['ID']}] gold={gold}  hits_in_top5={hits}")
    print(f"    Top-5: {top5}")
    print(f"    Desc: {row['enriched_desc'][:120]}...")

# ── 8. Comparison with SMET paper ────────────────────────────────
print(f"\n{'='*60}")
print(f"  COMPARISON WITH SMET PAPER")
print(f"{'='*60}")

smet_paper = {"Coverage Error": 13.96, "Ranking Loss": 0.05,
              "LRAP%": 53.77, "R@5%": 67.71}

all_modes = {
    "Full-text":  r_smet_raw,
    "Full+BRON":  r_smet_masked,
    "SMP":        r_smet_smp,
    "SMP+BRON":   r_smet_smp_m,
}

best_ce   = min(all_modes.values(), key=lambda x: x["coverage_error"])
best_rl   = min(all_modes.values(), key=lambda x: x["ranking_loss"])
best_lrap = max(all_modes.values(), key=lambda x: x["lrap_pct"])
best_r5   = max(all_modes.values(), key=lambda x: x["r5_pct"])

print(f"  {'Metric':<18} {'SMET Paper':>12} {'Ours (best)':>12} {'Delta':>10}")
print(f"  {'─'*18} {'─'*12} {'─'*12} {'─'*10}")
print(f"  {'Coverage Error':<18} {smet_paper['Coverage Error']:>12.2f} {best_ce['coverage_error']:>12.2f} {best_ce['coverage_error']-smet_paper['Coverage Error']:>+10.2f}")
print(f"  {'Ranking Loss':<18} {smet_paper['Ranking Loss']:>12.4f} {best_rl['ranking_loss']:>12.4f} {best_rl['ranking_loss']-smet_paper['Ranking Loss']:>+10.4f}")
print(f"  {'LRAP%':<18} {smet_paper['LRAP%']:>12.2f} {best_lrap['lrap_pct']:>12.2f} {best_lrap['lrap_pct']-smet_paper['LRAP%']:>+10.2f}")
print(f"  {'R@5%':<18} {smet_paper['R@5%']:>12.2f} {best_r5['r5_pct']:>12.2f} {best_r5['r5_pct']-smet_paper['R@5%']:>+10.2f}")

# ── 9. Store ──────────────────────────────────────────────────────
CFG._results["smet_raw"]        = r_smet_raw
CFG._results["smet_masked"]     = r_smet_masked
CFG._results["smet_smp_raw"]    = r_smet_smp
CFG._results["smet_smp_masked"] = r_smet_smp_m
CFG._results["smet_best_threshold"] = best_smet_thresh
CFG._results["smet_eval_rows"]  = len(df_smet_eval)
CFG._results["smet_total_rows"] = len(df_smet)
CFG._results["smet_label_coverage"] = {
    "in_space": len(in_space),
    "out_space": len(out_space),
    "in_space_tcodes": sorted(in_space),
    "out_space_tcodes": sorted(out_space),
}

print(f"\n✓ SMET evaluation complete. Ready for Cell 14 (final summary).")

SMET unique technique names  : 41
  Mapped to T-codes          : 40
  Unmapped names             : 1
    ['Indicator Removal on Host']
SMET T-codes (raw)           : 40
SMET T-codes (parent)        : 40
  In our 96-label space      : 29  ['T1005', 'T1007', 'T1016', 'T1040', 'T1055', 'T1078', 'T1083', 'T1110', 'T1176', 'T1195', 'T1211', 'T1213', 'T1491', 'T1498', 'T1499']
  Outside our space          : 11  ['T1059', 'T1068', 'T1136', 'T1189', 'T1190', 'T1203', 'T1204', 'T1485', 'T1518', 'T1529', 'T1531']

SMET rows with ≥1 in our space: 168 / 303
Overlap with training: 65
Descriptions enriched (CVSS/CWE prefix): 65 / 168

── Running SMET inference ──────────────────────────────
  probs: (168, 96)  labels: (168, 96)
  label density: 0.0113  avg labels/row: 1.08
  Sentence max-pool...
    Total sentences to process: 614 from 168 CVEs (avg 3.7 per CVE)

── SMET threshold sweep (full-text, no mask) ───────────
   Thresh     MiF1   CovErr   RkLoss    LRAP%     R@5%
     0.10   0.1276    21.9

In [39]:
# Cell 13b — SMET evaluated against BRON-derived labels (not human labels)

# ── 1. Build CWE → techniques mapping from BRON ──────────────────
# We need the same BRON chain used during training data creation
# Reconstruct from df_raw: for each CWE, collect all techniques it maps to

print("── Building CWE → technique mapping from training data ──")

# Method: parse the BRON graph directly
# We have bron_graph loaded (or need to reload)
try:
    _ = bron_graph
    print(f"  bron_graph already loaded: {len(bron_graph)} entries")
except NameError:
    print("  Loading BRON graph...")
    with open("BRON.json", encoding="utf-8") as f:
        bron_graph = json.load(f)
    print(f"  Loaded: {len(bron_graph)} entries")

# Build CWE → set of parent T-codes (same logic as training pipeline)
cwe_to_techniques = defaultdict(set)

# Parse BRON: find CWE nodes and their technique connections
# BRON structure varies — let's inspect first
sample_keys = list(bron_graph.keys())[:5]
print(f"  BRON top-level keys sample: {sample_keys}")

# Check if it's node-based or edge-based
sample_val = bron_graph[sample_keys[0]]
print(f"  Sample value type: {type(sample_val).__name__}")
if isinstance(sample_val, dict):
    print(f"  Sample value keys: {list(sample_val.keys())[:10]}")
elif isinstance(sample_val, list):
    print(f"  Sample value[0]: {sample_val[0] if sample_val else 'empty'}")

# Try to find CWE→technique paths
# Common BRON structures: 
#   node_id → {type, name, edges: [{target, type}]}
#   or adjacency: {cwe_id: [technique_ids]}

# Let's check what node types exist
node_types = Counter()
cwe_nodes = {}
technique_nodes = {}

if isinstance(sample_val, dict):
    for node_id, node_data in bron_graph.items():
        if isinstance(node_data, dict):
            ntype = node_data.get("original_id", node_data.get("datatype", ""))
            if "CWE-" in str(ntype) or "CWE-" in str(node_id):
                cwe_nodes[node_id] = node_data
            if "attack-pattern" in str(ntype) or "T" in str(node_id)[:2]:
                technique_nodes[node_id] = node_data
            # Count types
            dtype = node_data.get("datatype", "unknown")
            node_types[dtype] += 1

print(f"\n  Node types: {dict(node_types)}")
print(f"  CWE nodes found: {len(cwe_nodes)}")
print(f"  Technique nodes found: {len(technique_nodes)}")

if cwe_nodes:
    sample_cwe = list(cwe_nodes.items())[0]
    print(f"  Sample CWE node: {sample_cwe[0]} → {json.dumps(sample_cwe[1], indent=2)[:300]}")
if technique_nodes:
    sample_tech = list(technique_nodes.items())[0]
    print(f"  Sample technique node: {sample_tech[0]} → {json.dumps(sample_tech[1], indent=2)[:300]}")

── Building CWE → technique mapping from training data ──
  Loading BRON graph...


FileNotFoundError: [Errno 2] No such file or directory: 'BRON.json'